# CS-781 Capstone — Phase 1: parallel attention + Gated-DeltaNet hybrid decoder (LegalHymba-GDN), 160M words, on BOTH Kaggle GPUs, beam-search submission

**Architecture type: hybrid.** Inside every one of the 8 layers, a grouped-query **attention** branch (12 query heads / 3 KV heads, RoPE) and a causal **Gated DeltaNet** branch (4 heads x 128, a fixed-size recurrent memory updated with the delta rule) read the same normalised input **in parallel**; a learned **token-wise gate** (linear layer -> softmax over the two branches, initialised 50/50) mixes them for every token, then the dense feed-forward block follows. Idea: Hymba (Dong et al. 2024, arXiv 2411.13676) with Gated DeltaNet (Yang et al. 2024, arXiv 2412.06464) as the recurrent branch and our own token-wise gate. 126.9M parameters (the plain GQA decoder has 111.1M). Generation keeps a KV cache for attention plus a constant-size recurrent state per layer; the final step is beam search (width 4, no-repeat 4-grams).

**Data and recipe.** The same seeded selection and the same saved tokenizer as the other 160M-word models: **160M words (40M per source), one pass**. AdamW (betas 0.9/0.95, weight decay 0.1), peak lr 3e-4, 300 warm-up steps, cosine decay to 3e-5, gradient clipping 1.0, a final LayerNorm before the output head. This recipe differs from the plain GQA baseline's (constant lr, no clipping, no final norm), so a comparison with it is not perfectly controlled.

**Two GPUs.** Training runs as a separate `torchrun` job with one process per GPU (PyTorch DistributedDataParallel): each GPU processes `PER_GPU_BATCH` = 2 sequences per step and the gradients are averaged, so the **global batch is 4 sequences** (the baselines used 2) and there are about 22,500 steps instead of 45,000. Before training a 30-second NCCL check tests that the GPUs can talk to each other (falling back to `NCCL_P2P_DISABLE=1`, then to a single GPU). The speed of the GDN branch on a T4 and the 2-GPU scaling have **not been measured**: the log prints the real step speed and the time left after the first 100 steps. If the estimate is above ~10 h, lower `NEW_WORD_QUOTAS`. A checkpoint can only be resumed with the **same number of GPUs and batch settings**. The job saves a checkpoint every 1000 steps (pushed to the bucket every ~25 min), stops itself after `MAX_SESSION_HOURS` (11), and **continues if you run the notebook again from the top with the same settings**. `HYBRID_BRANCHES = "attn"` removes the GDN branch (a plain GQA decoder) for an ablation run. Run `QUICK_TEST = True` first (a few minutes).

## 1. Settings

In [1]:
# ----------------------------------------------------------------
# QUICK_TEST = True: a few-minute pipeline check (tiny slice, 60 batches, beam search on 16 test prompts, its own checkpoint name).
# Set it to False for the real run.
# ----------------------------------------------------------------
QUICK_TEST = False

# ---- Data: the SAME seeded selection as the other 160M-word runs (40M words from each source) --------------------
PREVIOUS_WORDS_PER_SOURCE = 0          # (nothing is skipped: this is a from-scratch run)
if QUICK_TEST:
    NEW_WORD_QUOTAS = {"supreme_court": 100_000, "high_courts": 100_000, "acts": 100_000, "tribunals": 100_000}
    MAX_TRAIN_BATCHES = 60
else:
    NEW_WORD_QUOTAS = {"supreme_court": 40_000_000, "high_courts": 40_000_000, "acts": 40_000_000, "tribunals": 40_000_000}
    MAX_TRAIN_BATCHES = None           # None = one pass over all the data
NUM_EPOCHS = 1
MAX_WORDS_FOR_TRAINING = sum(NEW_WORD_QUOTAS.values())

# ---- Architecture switch -------------------------------------------------------------------------------------------
HYBRID_BRANCHES = "both"               # "both" = attention + Gated DeltaNet in parallel in every layer; "attn" = attention only (ablation)

# ---- Optimizer / schedule ------------------------------------------------------------------------------------------
VARIANT = dict(optimizer="adamw", betas=(0.9, 0.95), weight_decay=0.1, grad_clip=1.0)     # AdamW+ (the sweep's safe choice)
FINAL_NORM = True                      # a LayerNorm before the output head
PEAK_LR = 3e-4
WARMUP_STEPS = 10 if QUICK_TEST else 300
DECAY_START_FRACTION = 0.0             # 0.0 = cosine decay from the end of the warm-up (what the sweep used). 0.8 = flat for 80% of the run, then decay.
MIN_LR_FRACTION = 0.1                  # decay down to PEAK_LR * this at the last batch
MAX_SESSION_HOURS = 11.0               # after this long in ONE session: save, push and stop; re-run the notebook to continue
INIT_SEED = 0                          # model initialisation seed (same as in the sweep)

VOCAB_SIZE = 40000
CONTEXT_LEN = 2048
BATCH_SIZE = 2                         # (only used by the notebook's own evaluation / generation cells)
LEARNING_RATE = PEAK_LR
TEMPERATURE = 0.3
NORM_TYPE = "layernorm"; NORM_POSITION = "pre"; OPTIMIZER_TYPE = "adamw"      # (read by the shared model cells; the real optimizer is VARIANT above)

RUN_TAG = "hybrid_gdn_160Mw"                  # run-log name
mode_tag = "quicktest" if QUICK_TEST else "full"
TOKENIZER_RUN_TAG = "160Mw_1ep"               # the saved 160M-word tokenizer (the same one the other 160M-word models use)
TOKENIZER_WORD_COUNT = 160_000_000
REQUIRE_SAVED_TOKENIZER = True

CHECKPOINT_NAME = f"phase1_decoder_only_hybrid_gdn_attn_160Mw_2gpu_{mode_tag}_checkpoint.pt" if HYBRID_BRANCHES == "both" else f"phase1_decoder_only_gdn_ablation_attn_only_160Mw_2gpu_{mode_tag}_checkpoint.pt"
CHECKPOINT_PATH = "/kaggle/working/" + CHECKPOINT_NAME

# ---- Multi-GPU training -------------------------------------------------------------------------------------------
NUM_GPUS = None                        # None = use every GPU Kaggle gives this session (2 with "GPU T4 x2"); 1 = force one GPU
PER_GPU_BATCH = 2                      # sequences per GPU per step. Global batch = NUM_GPUS x PER_GPU_BATCH (4 on two GPUs). Keep it the same in every session of this run.
DATA_ORDER_SEED = 2000                 # fixed: a resumed run sees the same batches
SYNC_REPLICAS_EVERY = 5 if QUICK_TEST else 500    # insurance: copy GPU 0's weights to the other GPU this often
PUSH_MIN_MINUTES = 0 if QUICK_TEST else 25        # the checkpoint (saved every 1000 steps) is pushed to the bucket at most this often


## 2. Make sure the working folder exists

In [2]:
import os
os.makedirs("/kaggle/working", exist_ok=True)
print("Working folder ready:", os.path.exists("/kaggle/working"))

Working folder ready: True


## 3. Imports and Hugging Face login

In [3]:
!pip install -q -U "huggingface_hub>=1.5"   # storage buckets need huggingface_hub >= 1.5
import json
import os
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 15.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
Using device: cuda


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 3b. Model storage (the bucket) and run log

In [4]:
from huggingface_hub import HfApi, create_repo, create_bucket, batch_bucket_files, download_bucket_files, list_bucket_tree, hf_hub_download
import shutil

HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
hf_api = HfApi(token=HF_WRITE_TOKEN)

# ---- Where models live: a Hugging Face STORAGE BUCKET (not a repo) -------------------------------------------
# A repo is a git repository: every overwritten checkpoint stays in its history, so a few 3 GB models quickly
# cost tens of GB. A bucket is plain mutable storage: overwriting a file REPLACES it (no history), so each model
# costs exactly its own size. Needs huggingface_hub >= 1.5 (installed in Section 3).
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # the OLD repo: only READ, as a fallback for models not moved yet
create_bucket(HF_BUCKET, private=True, exist_ok=True, token=HF_WRITE_TOKEN)


def list_model_files():
    """Names of every file in the bucket (empty list if the bucket cannot be listed)."""
    try:
        return [f.path for f in list_bucket_tree(HF_BUCKET, recursive=True, token=HF_WRITE_TOKEN)
                if getattr(f, "type", "file") == "file"]
    except Exception as e:
        print(f"(could not list the bucket: {type(e).__name__})")
        return []


def upload_model_file(local_path, name=None, quiet=False):
    """Upload local_path to the bucket as `name` (default: its own filename). Replaces any older copy."""
    name = name or os.path.basename(local_path)
    batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
    if not quiet:
        print(f"Uploaded {name} to bucket {HF_BUCKET}")


def fetch_model_file(name, local_path=None):
    """Download `name` from the bucket (falling back to the old repo) to local_path; returns the local path.
    Raises FileNotFoundError if it is in neither place."""
    local_path = local_path or f"/kaggle/working/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        if not os.path.exists(part):
            raise FileNotFoundError(name)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    try:   # fallback: the old repo (read-only)
        p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
        shutil.copy(p, local_path)
        print(f"(found {name} only in the old repo {HF_CHECKPOINT_REPO}; new saves go to the bucket)")
        return local_path
    except Exception:
        raise FileNotFoundError(f"{name} is in neither the bucket {HF_BUCKET} nor the repo {HF_CHECKPOINT_REPO}")


def push_checkpoint_to_hub(local_path, quiet=False):
    """Upload a checkpoint, the tokenizer, or the run log to the bucket under its own filename (which already
    encodes the full config -- see Settings). Each upload REPLACES the previous version, and nothing is kept."""
    upload_model_file(local_path, quiet=quiet)


def pull_from_hub(local_path):
    """Quietly fetch local_path's file (by basename) if it is not on disk yet. True if it exists afterwards."""
    if os.path.exists(local_path):
        return True
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled {os.path.basename(local_path)} from the Hub")
        return True
    except Exception:
        return False


def try_pull_checkpoint_from_hub(local_path):
    """If no LOCAL checkpoint exists yet (fresh Kaggle session / other account), fetch a matching one."""
    if os.path.exists(local_path):
        return
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled matching checkpoint from the Hub: {os.path.basename(local_path)}")
    except Exception as e:
        print(f"No matching checkpoint on the Hub yet ({type(e).__name__}) -- starting fresh.")


print(f"Model storage configured -> bucket {HF_BUCKET}")

# ---- Run log: a running record of what this project did, for the final report ----
# Every important step appends one JSON line (time, what happened, the numbers) to
# RUN_LOG_PATH. The file is pushed to the Hub together with the checkpoints and pulled
# back at the start of a new session, so a run that spans several sessions (or both
# Kaggle accounts) keeps ONE continuous log. Read it with: print(open(RUN_LOG_PATH).read())
import time as _time, datetime as _dt
RUN_LOG_PATH = f"/kaggle/working/run_log_gqa_kvcache_{RUN_TAG}_{mode_tag}.jsonl"
pull_from_hub(RUN_LOG_PATH)


def log_event(kind, **fields):
    """Append one event to the run log (never raises: logging must not stop a training run)."""
    try:
        record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")
    except Exception as e:
        print(f"(run log write failed: {type(e).__name__})")


def sync_run_log():
    try:
        push_checkpoint_to_hub(RUN_LOG_PATH, quiet=True)
    except Exception as e:
        print(f"(run log push failed: {type(e).__name__})")


log_event("session_start", quick_test=QUICK_TEST, vocab_size=VOCAB_SIZE, context_len=CONTEXT_LEN,
          batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
          max_words=MAX_WORDS_FOR_TRAINING, norm=f"{NORM_TYPE}/{NORM_POSITION}", optimizer=OPTIMIZER_TYPE,
          device=device, gpu=(torch.cuda.get_device_name(0) if device == "cuda" else None))
print(f"Run log: {RUN_LOG_PATH}")

Model storage configured -> bucket kavincg23/cs781-models
Run log: /kaggle/working/run_log_gqa_kvcache_hybrid_gdn_160Mw_full.jsonl


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 4. Dataset names and the test prompts

In [5]:
!pip install -q ijson
import ijson

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.5 MB/s eta 0:00:00


In [6]:
TEXT_FIELD = "text"  # <-- change this if needed, based on what you saw above

In [7]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                  repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_prompts = {str(doc_id): record[TEXT_FIELD] for doc_id, record in test_data.items()}

print(f"Loaded {len(test_prompts)} test prompts")
first_id = list(test_prompts.keys())[0]
print(f"Example prompt (id={first_id}), word count = {len(test_prompts[first_id].split())}")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Loaded 1000 test prompts
Example prompt (id=1), word count = 512


## 5. The training documents (and the held-out evaluation documents)

The same seeded selection as the existing 160M-word run: a seeded random order over each source file, the **held-out** documents first (never trained on), then the training documents until 40M words per source. With `PREVIOUS_WORDS_PER_SOURCE = 0` the training documents are exactly the ones the existing MoE model was trained on.

In [8]:
import array, random, time
SAMPLING_SEED = 42      # MUST match the base run: same seed -> same shuffle -> same held-out and previous documents
HELDOUT_WORDS = {'acts': 500000, 'high_courts': 500000, 'supreme_court': 2000000, 'tribunals': 500000}
HELDOUT_MIN_DOC_WORDS = 640
HELDOUT_MAX_WORDS_PER_DOC = 6000
HELDOUT_FILES = {}
CORPUS_FILE = "/kaggle/working/continue_training_corpus.txt"
print("Streaming through the training files...")

word_count = 0
new_words_by_source = {}
selection_debug = {}
with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
    for filename in TRAIN_FILES:
        short = os.path.basename(filename)[len("train_"):-len(".json")]
        new_quota = NEW_WORD_QUOTAS.get(short, 0)
        if new_quota <= 0 and short != "supreme_court":
            continue        # nothing to train on from this source (Supreme Court is always read: it supplies the held-out documents)
        local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)

        t0 = time.time()
        doc_lengths = array.array("l")
        with open(local_path, "r", encoding="utf-8") as f:
            for record in ijson.items(f, "item"):
                doc_lengths.append(len(record[TEXT_FIELD].split()))
        print(f"  {filename}: {len(doc_lengths):,} documents, {sum(doc_lengths):,} words (counted in {time.time() - t0:.0f}s)")

        order = list(range(len(doc_lengths)))
        random.Random(SAMPLING_SEED).shuffle(order)

        # 1) held-out documents (identical to the base run)
        held_target = HELDOUT_WORDS.get(short, 0)
        heldout_ids, held_words = set(), 0
        for i in order:
            if held_words >= held_target:
                break
            if doc_lengths[i] >= HELDOUT_MIN_DOC_WORDS:
                heldout_ids.add(i)
                held_words += min(doc_lengths[i], HELDOUT_MAX_WORDS_PER_DOC)

        # 2) the documents the base run trained on (identical selection, quota PREVIOUS_WORDS_PER_SOURCE)
        previous_ids, planned = set(), 0
        for i in order:
            if planned >= PREVIOUS_WORDS_PER_SOURCE:
                break
            if i in heldout_ids:
                continue
            keep = min(doc_lengths[i], PREVIOUS_WORDS_PER_SOURCE - planned)
            if keep > 0:
                previous_ids.add(i)
                planned += keep

        # 3) NEW documents: the next ones in the same order
        chosen, planned = {}, 0
        for i in order:
            if planned >= new_quota:
                break
            if i in heldout_ids or i in previous_ids:
                continue
            keep = min(doc_lengths[i], new_quota - planned)
            if keep > 0:
                chosen[i] = keep
                planned += keep
        available = sum(doc_lengths) - sum(doc_lengths[i] for i in heldout_ids | previous_ids)
        if planned < new_quota:
            print(f"  NOTE: {short} only has {planned:,} unused words (asked for {new_quota:,}); using all of them.")
        del order
        selection_debug[short] = (heldout_ids, previous_ids, set(chosen))     # (kept so the selection can be inspected)

        write_held = heldout_ids if short == "supreme_court" else set()     # only Supreme Court held-out text is used for evaluation
        source_word_count = 0
        remaining_docs, remaining_held = len(chosen), len(write_held)
        held_f = None
        if write_held:
            heldout_path = os.path.join(os.path.dirname(CORPUS_FILE), f"heldout_{short}.txt")
            held_f = open(heldout_path, "w", encoding="utf-8")
            HELDOUT_FILES[short] = heldout_path
        with open(local_path, "r", encoding="utf-8") as f:
            for i, record in enumerate(ijson.items(f, "item")):
                keep = chosen.get(i)
                if keep is None and i not in write_held:
                    continue
                doc_words = record[TEXT_FIELD].split()
                if keep is not None:
                    out_f.write(" ".join(doc_words[:keep]) + " ")
                    source_word_count += min(keep, len(doc_words))
                    remaining_docs -= 1
                else:
                    held_f.write(" ".join(doc_words[:HELDOUT_MAX_WORDS_PER_DOC]) + "\n")
                    remaining_held -= 1
                if remaining_docs == 0 and remaining_held == 0:
                    break
        if held_f is not None:
            held_f.close()
        word_count += source_word_count
        new_words_by_source[short] = source_word_count
        print(f"  ...{short}: {source_word_count:,} NEW words from {len(chosen):,} documents (quota {new_quota:,}, unused words in the file: {available:,}); "
              f"skipped {len(previous_ids):,} documents of the previous run; held out {len(heldout_ids):,} documents")

MAX_WORDS_FOR_TRAINING = word_count
print(f"\nNEW training words: {word_count:,}  (written to {CORPUS_FILE})   by source: {new_words_by_source}")
log_event("continue_data", new_words=word_count, by_source=new_words_by_source, previous_per_source=PREVIOUS_WORDS_PER_SOURCE)


Streaming through the training files...


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_acts.json: 18,826 documents, 119,638,348 words (counted in 10s)
  ...acts: 40,000,000 NEW words from 6,615 documents (quota 40,000,000, unused words in the file: 118,684,097); skipped 0 documents of the previous run; held out 140 documents


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (4 outputs left out)

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_tribunals.json: 869,132 documents, 975,917,446 words (counted in 91s)
  ...tribunals: 40,000,000 NEW words from 35,848 documents (quota 40,000,000, unused words in the file: 975,346,913); skipped 0 documents of the previous run; held out 273 documents

NEW training words: 160,000,000  (written to /kaggle/working/continue_training_corpus.txt)   by source: {'acts': 40000000, 'high_courts': 40000000, 'supreme_court': 40000000, 'tribunals': 40000000}


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 6. The saved tokenizer

In [9]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)


# The tokenizer depends only on the training corpus, VOCAB_SIZE and BPE_MIN_WORD_FREQ, so its file name records
# exactly those (the corpus is fixed by the sampling seed, the quotas and the word count). If a file with this name is
# on disk or on the Hub it is loaded (skipping the slow pure-Python training); otherwise it is trained, saved and pushed.
TOKENIZER_PATH = (f"/kaggle/working/a1bpe_tokenizer_{TOKENIZER_RUN_TAG}_{mode_tag}_v{VOCAB_SIZE}_{TOKENIZER_WORD_COUNT}w"
                  f"_seed{SAMPLING_SEED}_minf{BPE_MIN_WORD_FREQ}.json")
t_start = time.time()
if pull_from_hub(TOKENIZER_PATH):
    print(f"Loading the saved tokenizer {os.path.basename(TOKENIZER_PATH)} (skipping training)...")
    bpe_tokenizer = BPETokenizer.load(TOKENIZER_PATH)
    tokenizer_source = "loaded"
else:
    if REQUIRE_SAVED_TOKENIZER:
        raise RuntimeError(f"The saved tokenizer {os.path.basename(TOKENIZER_PATH)} was not found locally or in the bucket {HF_BUCKET}. "
                           "This notebook only decodes with the trained model, so it needs the SAME tokenizer the model was trained with. "
                           "Check the HF token / bucket and the tag/word count in Settings.")
    print("Training the BPE tokenizer (our own implementation, pure Python)...")
    bpe_tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE).fit_file(CORPUS_FILE, min_word_freq=BPE_MIN_WORD_FREQ)
    bpe_tokenizer.save(TOKENIZER_PATH, meta={"word_count": word_count, "sampling_seed": SAMPLING_SEED,
                                             "min_word_freq": BPE_MIN_WORD_FREQ, "requested_vocab": VOCAB_SIZE})
    try:
        push_checkpoint_to_hub(TOKENIZER_PATH)
    except Exception as e:
        print(f"(could not push the tokenizer to the Hub: {type(e).__name__} -- it is still saved locally)")
    tokenizer_source = "trained"
print(f"Tokenizer {tokenizer_source} in {time.time() - t_start:.0f}s")
log_event("tokenizer", source=tokenizer_source, seconds=round(time.time() - t_start), vocab_size=bpe_tokenizer.get_vocab_size(),
          merges=len(bpe_tokenizer.merges), words=word_count, file=os.path.basename(TOKENIZER_PATH))


def encode(text):
    """Turn a string into a list of BPE token IDs."""
    return bpe_tokenizer.encode(text)


def decode(ids):
    """Turn a list of BPE token IDs back into a string."""
    return bpe_tokenizer.decode(ids)


print("Vocabulary size:", bpe_tokenizer.get_vocab_size())
if bpe_tokenizer.get_vocab_size() < VOCAB_SIZE:
    print(f"NOTE: fewer than VOCAB_SIZE={VOCAB_SIZE} merges were possible; the model uses the real size above.")

# Encode the NEW training corpus (streamed from disk). all_word_ids holds BPE-token ids.
t_start = time.time()
all_word_ids = bpe_tokenizer.encode_file(CORPUS_FILE)
print(f"{word_count:,} words became {len(all_word_ids):,} BPE tokens "
      f"({len(all_word_ids) / max(1, word_count):.2f} tokens per word) in {time.time() - t_start:.0f}s")
log_event("encoded_corpus", words=word_count, tokens=len(all_word_ids), tokens_per_word=round(len(all_word_ids) / max(1, word_count), 3),
          seconds=round(time.time() - t_start))


(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json only in the old repo kavincg23/cs781-checkpoints; new saves go to the bucket)
Pulled a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json from the Hub
Loading the saved tokenizer a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json (skipping training)...
Tokenizer loaded in 1s
Vocabulary size: 40000


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


160,000,000 words became 183,754,833 BPE tokens (1.15 tokens per word) in 173s


## 7. Model code (identical to the training notebook)

In [10]:
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

### Optimizers: AdamW+ and Muon

The same code as the sweep (tested there): **Muon** (Nesterov momentum 0.95, 5 Newton-Schulz steps in fp16 on the T4's tensor cores, update scaled by 0.2 * sqrt(max(rows, cols)) so it has AdamW's size, decoupled weight decay 0.1) for the hidden 2-D matrices (attention projections, expert and FFN matrices); **AdamW** (betas 0.9/0.95, weight decay 0.1 on matrices only) for everything else. `MultiOptimizer` lets the loop treat both as one and saves/loads both states.

In [11]:
import math
NS_DTYPE = torch.float16 if device == "cuda" else torch.float32     # Newton-Schulz in fp16 uses the T4's tensor cores; the matrices are normalised so fp16 range is fine


def zeropower_batched(G, steps=5, eps=1e-7):
    """Newton-Schulz orthogonalisation of a BATCH of same-shaped matrices G: (B, m, n) (Keller Jordan's quintic coefficients)."""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.to(NS_DTYPE)
    tall = X.size(-2) > X.size(-1)
    if tall:
        X = X.transpose(-2, -1)
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + eps)
    for _ in range(steps):
        A = X @ X.transpose(-2, -1)
        X = a * X + (b * A + c * (A @ A)) @ X
    if tall:
        X = X.transpose(-2, -1)
    return X


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices: momentum (Nesterov) -> orthogonalise -> scale -> decoupled weight decay.
    The update is scaled by 0.2 * sqrt(max(rows, cols)) so its RMS matches AdamW's (Moonlight, Liu et al. 2025);
    that lets Muon reuse AdamW's learning rate and weight decay. Same-shaped matrices are orthogonalised together in one batched call."""

    def __init__(self, params, lr, momentum=0.95, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, mu, wd = group["lr"], group["momentum"], group["weight_decay"]
            by_shape = {}
            for p in group["params"]:
                if p.grad is None:
                    continue
                st = self.state[p]
                if "momentum_buffer" not in st:
                    st["momentum_buffer"] = torch.zeros_like(p)
                buf = st["momentum_buffer"]
                buf.mul_(mu).add_(p.grad)
                upd = p.grad.add(buf, alpha=mu)                      # Nesterov
                by_shape.setdefault(tuple(p.shape), []).append((p, upd))
            for shape, items in by_shape.items():
                O = zeropower_batched(torch.stack([u for _, u in items]), steps=group["ns_steps"])
                scale = 0.2 * math.sqrt(max(shape))
                for (p, _), o in zip(items, O):
                    if wd:
                        p.mul_(1 - lr * wd)
                    p.add_(o.to(p.dtype), alpha=-lr * scale)


class MultiOptimizer:
    """Lets the loop treat several optimizers as one."""

    def __init__(self, opts):
        self.opts = opts

    @property
    def param_groups(self):
        return [g for o in self.opts for g in o.param_groups]

    def zero_grad(self):
        for o in self.opts:
            o.zero_grad(set_to_none=True)

    def step(self):
        for o in self.opts:
            o.step()

    def state_dict(self):
        return [o.state_dict() for o in self.opts]

    def load_state_dict(self, states):
        for o, s in zip(self.opts, states):
            o.load_state_dict(s)


def is_muon_matrix(name, p):
    """Hidden 2-D weight matrices (attention projections, expert / FFN matrices). NOT the embedding table, the output head, the router or norms."""
    return p.ndim == 2 and name.startswith("blocks.") and "router" not in name and "norm" not in name


def build_optimizer(model, v, peak_lr):
    kind = v["optimizer"]
    betas = tuple(v.get("betas", (0.9, 0.95)))
    wd = v.get("weight_decay", 0.1)
    named = list(model.named_parameters())
    if kind == "muon":
        muon_params = [p for n, p in named if is_muon_matrix(n, p)]
        rest = [(n, p) for n, p in named if not is_muon_matrix(n, p)]
    else:
        muon_params, rest = [], named
    if v.get("decay_all", False):
        decay = [p for _, p in rest]; no_decay = []
    else:
        decay = [p for n, p in rest if p.ndim >= 2 and "token_emb" not in n and "router" not in n]
        no_decay = [p for n, p in rest if not (p.ndim >= 2 and "token_emb" not in n and "router" not in n)]
    groups = [dict(params=decay, weight_decay=wd)]
    if no_decay:
        groups.append(dict(params=no_decay, weight_decay=0.0))
    adam = torch.optim.AdamW(groups, lr=peak_lr, betas=betas, fused=(device == "cuda"))
    if kind == "muon":
        mu = Muon(muon_params, lr=peak_lr, momentum=v.get("muon_momentum", 0.95), weight_decay=wd, ns_steps=v.get("ns_steps", 5))
        return MultiOptimizer([mu, adam]), dict(muon_tensors=len(muon_params), muon_params=sum(p.numel() for p in muon_params),
                                                adamw_params=sum(p.numel() for g in groups for p in g["params"]))
    return MultiOptimizer([adam]), dict(muon_tensors=0, muon_params=0, adamw_params=sum(p.numel() for g in groups for p in g["params"]))




def build_optimizer(model, v, peak_lr):
    """v: a variant dict (see VARIANT in Settings). Returns (optimizer, info)."""
    kind = v["optimizer"]
    betas = tuple(v.get("betas", (0.9, 0.95)))
    wd = v.get("weight_decay", 0.1)
    named = list(model.named_parameters())
    if kind == "muon":
        muon_params = [p for n, p in named if is_muon_matrix(n, p)]
        rest = [(n, p) for n, p in named if not is_muon_matrix(n, p)]
    else:
        muon_params, rest = [], named
    decay = [p for n, p in rest if p.ndim >= 2 and "token_emb" not in n and "router" not in n]
    no_decay = [p for n, p in rest if not (p.ndim >= 2 and "token_emb" not in n and "router" not in n)]
    groups = [dict(params=decay, weight_decay=wd)]
    if no_decay:
        groups.append(dict(params=no_decay, weight_decay=0.0))
    adam = torch.optim.AdamW(groups, lr=peak_lr, betas=betas, fused=(device == "cuda"))
    if kind == "muon":
        mu = Muon(muon_params, lr=peak_lr, momentum=v.get("muon_momentum", 0.95), weight_decay=wd, ns_steps=v.get("ns_steps", 5))
        return MultiOptimizer([mu, adam]), dict(muon_tensors=len(muon_params), muon_params=sum(p.numel() for p in muon_params),
                                                adamw_params=sum(p.numel() for g in groups for p in g["params"]))
    return MultiOptimizer([adam]), dict(muon_tensors=0, muon_params=0, adamw_params=sum(p.numel() for g in groups for p in g["params"]))


In [12]:
# ---- Settings specific to this notebook ------------------------------
# (GDN_* sizes are defined with the Gated DeltaNet code below)
# ---- Mixture-of-Experts FFN (DeepSeekMoE style) -------------------------------------------------------------
MOE_N_ROUTED = 16          # routed experts per MoE layer (fine-grained: each is only 1/4 the width of the dense FFN)
MOE_N_SHARED = 1           # shared experts: ALWAYS applied to every token (they hold the common knowledge)
MOE_TOP_K = 3              # routed experts used per token. Active FFN width = (1 shared + 3 routed) x 768 = 3072 = the dense FFN's width,
                           # so the compute per token matches the dense GQA run; the TOTAL FFN capacity is 17 x 768 = 4.25x larger.
MOE_D_EXPERT = 768         # hidden width of one expert
MOE_DENSE_FIRST_LAYERS = 99 # >= number of layers: NO layer uses the MoE FFN here (all dense). (The MoE code below is kept only so that the shared helper functions exist.)
MOE_BIAS_RATE = 0.001      # aux-loss-free load balancing (DeepSeek-V3): per step, a per-expert selection bias moves by this much
                           # towards the under-used experts. The bias only affects WHICH experts are chosen, never the output weights.

GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class ExpertMLP(nn.Module):
    """One expert: the same Linear -> GELU -> Linear as the dense FFN, just narrower."""

    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up = nn.Linear(d_model, d_hidden)
        self.down = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class MoEFFN(nn.Module):
    """DeepSeekMoE feed-forward layer, written from scratch.

        y = sum_s Shared_s(x)  +  sum_{e in TopK(x)} g_e(x) * Routed_e(x)

    * router: affinity s_e = sigmoid(w_e . x) for every routed expert e (DeepSeek-V3 uses sigmoid, not softmax)
    * selection: the top-k experts by (s_e + bias_e); the bias is a NON-learned buffer used only for choosing
    * gates: the selected s_e, renormalised to sum to 1 (so the routed part has a fixed scale)
    * load balancing WITHOUT an auxiliary loss: in training, after each forward pass, bias_e moves by +-MOE_BIAS_RATE
      towards the under-loaded experts (overloaded: down, underloaded: up). Nothing is dropped (no capacity limit).
    Every token is processed independently, so the KV-cache decode path needs no change."""

    def __init__(self, d_model, n_routed, n_shared, top_k, d_expert, bias_rate):
        super().__init__()
        assert 1 <= top_k <= n_routed
        self.n_routed, self.top_k, self.bias_rate = n_routed, top_k, bias_rate
        self.router = nn.Linear(d_model, n_routed, bias=False)
        nn.init.normal_(self.router.weight, std=0.02)
        self.shared = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_shared)])
        self.experts = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_routed)])
        self.register_buffer("select_bias", torch.zeros(n_routed))      # saved in the checkpoint (it is part of the trained router)
        self.last_load = None                                           # tokens routed to each expert in the latest training step

    def forward(self, x):
        shape = x.shape
        flat = x.reshape(-1, shape[-1])                                  # (N, d): the router does not care about batch/position
        affinity = torch.sigmoid(self.router(flat).float())              # (N, E), fp32 even under autocast
        _, chosen = (affinity + self.select_bias).topk(self.top_k, dim=-1)       # (N, k): selection uses the bias ...
        gate = affinity.gather(1, chosen)                                # ... but the gate values do not
        gate = gate / gate.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(flat)
        for expert in self.shared:
            out = out + expert(flat)
        for e, expert in enumerate(self.experts):
            token_idx, slot = (chosen == e).nonzero(as_tuple=True)       # the tokens that picked expert e, and in which of their k slots
            if token_idx.numel() == 0:
                continue
            y = expert(flat[token_idx]) * gate[token_idx, slot].unsqueeze(-1).to(flat.dtype)
            out.index_add_(0, token_idx, y.to(out.dtype))
        if self.training:
            with torch.no_grad():
                load = torch.bincount(chosen.reshape(-1), minlength=self.n_routed).float()
                self.last_load = load
                self.select_bias += self.bias_rate * torch.sign(load.mean() - load)
        return out.reshape(shape)


def moe_load_stats(model):
    """[(layer index, busiest expert's load / the average (1.0 = perfectly balanced), number of experts with < 10% of an even share)]
    for every MoE layer, from the latest training step."""
    rows = []
    for i, blk in enumerate(model.blocks):
        if isinstance(blk.ff, MoEFFN) and blk.ff.last_load is not None:
            l = blk.ff.last_load.cpu()
            rows.append((i, round((l.max() / l.mean()).item(), 3), int((l < 0.1 * l.mean()).sum().item())))
    return rows


def moe_load_report(model):
    rows = moe_load_stats(model)
    if not rows:
        print("MoE load report: no training step has run in this session yet.")
        return
    print("MoE load balance (max expert load / mean, experts with <10% of an even share):")
    print("  " + "   ".join(f"layer {i}: {r:.2f}x / {d}" for i, r, d in rows))


def count_active_params(model):
    """Parameters used per token: everything except the routed experts that are NOT selected (top_k of n_routed are used)."""
    total = sum(p.numel() for p in model.parameters())
    inactive = 0
    for blk in model.blocks:
        if isinstance(blk.ff, MoEFFN):
            per_expert = sum(p.numel() for p in blk.ff.experts[0].parameters())
            inactive += per_expert * (blk.ff.n_routed - blk.ff.top_k)
    return total, total - inactive


import math
GDN_HEADS = 4          # heads per Gated DeltaNet layer
GDN_HEAD_DIM = 128     # key dim = value dim per head (inner width = 4 x 128 = 512)
GDN_CONV = 4           # width of the short causal depthwise convolution over q, k, v
GDN_CHUNK = 64         # tokens per chunk in the chunkwise form


def _l2norm(x, eps=1e-6):
    return x / (x.norm(dim=-1, keepdim=True) + eps)


def gdn_recurrent(q, k, v, log_alpha, beta, state=None):
    """REFERENCE (slow, one token at a time). q,k: (B,H,L,dk); v: (B,H,L,dv); log_alpha, beta: (B,H,L); state: (B,H,dv,dk) or None.
    Returns outputs (B,H,L,dv) and the final state."""
    B, H, L, dk = q.shape
    dv = v.shape[-1]
    S = q.new_zeros(B, H, dv, dk) if state is None else state
    outs = []
    for t in range(L):
        kt, vt, qt = k[:, :, t], v[:, :, t], q[:, :, t]
        a = log_alpha[:, :, t].exp()[..., None, None]
        b = beta[:, :, t][..., None, None]
        Sk = torch.einsum("bhvk,bhk->bhv", S, kt)
        S = a * (S - b * Sk.unsqueeze(-1) * kt.unsqueeze(-2)) + b * vt.unsqueeze(-1) * kt.unsqueeze(-2)
        outs.append(torch.einsum("bhvk,bhk->bhv", S, qt))
    return torch.stack(outs, dim=2), S


def gdn_chunkwise(q, k, v, log_alpha, beta, state=None, chunk=64):
    """Same function as gdn_recurrent, computed `chunk` tokens at a time.
    Inside a chunk (c_i = cumulative log-decay, gamma_i = exp(c_i)) the state is  S_i = gamma_i S0 + sum_{j<=i} (gamma_i/gamma_j) d_j k_j^T,
    where the 'corrected values' d_j solve a small lower-triangular system:  (I + A) D = beta*V - (beta*gamma*K) S0^T ,
    A_ij = beta_i (gamma_i/gamma_j) k_i.k_j for j < i.   With T = (I+A)^-1:  U = T (beta V),  W = T (beta gamma K),  D = U - W S0^T.
    Then   O = diag(gamma) Q S0^T + ((Q K^T) * Lambda) D ,   S_end = gamma_C S0 + D^T diag(gamma_C/gamma) K.
    Decay ratios are built as exp(c_i - c_j) AFTER masking j > i (never a division of cumulative products), so nothing over/underflows."""
    B, H, L, dk = q.shape
    dv = v.shape[-1]
    pad = (-L) % chunk
    if pad:   # right-pad with 'do nothing' tokens (k = v = q = 0, beta = 0, alpha = 1): the state is untouched by them
        q, k, v = (F.pad(x, (0, 0, 0, pad)) for x in (q, k, v))
        log_alpha, beta = (F.pad(x, (0, pad)) for x in (log_alpha, beta))
    N = (L + pad) // chunk
    q, k, v = (x.reshape(B, H, N, chunk, x.shape[-1]) for x in (q, k, v))
    la, be = (x.reshape(B, H, N, chunk) for x in (log_alpha, beta))
    c = la.cumsum(dim=-1)                                                         # (B,H,N,C) inclusive cumulative log-decay
    on_or_below = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))
    strictly_below = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device), diagonal=-1)
    lam = (c.unsqueeze(-1) - c.unsqueeze(-2)).masked_fill(~on_or_below, float("-inf")).exp()   # gamma_i/gamma_j for j<=i, else 0
    A = (be.unsqueeze(-1) * (k @ k.transpose(-1, -2)) * lam).masked_fill(~strictly_below, 0.0)
    eye = torch.eye(chunk, dtype=q.dtype, device=q.device)
    T = torch.linalg.solve_triangular(eye + A, eye.expand_as(A), upper=False)      # (I + A)^-1
    u = T @ (be.unsqueeze(-1) * v)                                                 # (B,H,N,C,dv)
    w = T @ ((be * c.exp()).unsqueeze(-1) * k)                                     # (B,H,N,C,dk)
    qk = (q @ k.transpose(-1, -2)) * lam                                           # (B,H,N,C,C)
    q_g = c.exp().unsqueeze(-1) * q
    c_last = c[..., -1]                                                            # (B,H,N)
    k_dec = (c_last.unsqueeze(-1) - c).exp().unsqueeze(-1) * k
    S = q.new_zeros(B, H, dv, dk) if state is None else state
    outs = []
    for n in range(N):
        St = S.transpose(-1, -2)                                                   # (B,H,dk,dv)
        d = u[:, :, n] - w[:, :, n] @ St                                           # corrected values (B,H,C,dv)
        outs.append(q_g[:, :, n] @ St + qk[:, :, n] @ d)
        S = c_last[:, :, n].exp()[..., None, None] * S + d.transpose(-1, -2) @ k_dec[:, :, n]
    o = torch.stack(outs, dim=2).reshape(B, H, N * chunk, dv)[:, :, :L]
    return o, S


class GatedDeltaNetMixer(nn.Module):
    """The sequence-mixing part of a Gated DeltaNet layer (the block wraps it with the norm, residual and MLP).
    x -> q,k,v projections -> short causal depthwise conv + SiLU -> L2-normalised q,k -> delta-rule memory (gate alpha, step beta)
      -> per-head RMSNorm -> times SiLU(output gate) -> output projection.   Always computed in fp32 (the recurrence is the numerically
    sensitive part; it is a small share of the compute)."""

    def __init__(self, d_model, n_heads=GDN_HEADS, head_dim=GDN_HEAD_DIM, conv_width=GDN_CONV, chunk=GDN_CHUNK):
        super().__init__()
        self.n_heads, self.head_dim, self.conv_width, self.chunk = n_heads, head_dim, conv_width, chunk
        inner = n_heads * head_dim
        self.inner = inner
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.g_proj = nn.Linear(d_model, inner, bias=False)          # output gate
        self.o_proj = nn.Linear(inner, d_model, bias=False)
        self.a_proj = nn.Linear(d_model, n_heads, bias=False)        # -> decay alpha_t (per head)
        self.b_proj = nn.Linear(d_model, n_heads, bias=False)        # -> step size beta_t (per head)
        self.conv = nn.Conv1d(3 * inner, 3 * inner, conv_width, groups=3 * inner, bias=False)   # depthwise, applied to [q;k;v]
        self.out_norm = RMSNorm(head_dim)
        # decay parameterisation (Mamba-2 style): alpha = exp(-exp(A_log) * softplus(a_proj(x) + dt_bias)), A in [1,16], dt in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(n_heads).uniform_(1.0, 16.0)))
        dt = torch.exp(torch.rand(n_heads) * (math.log(0.1) - math.log(0.001)) + math.log(0.001))
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))        # inverse softplus of dt

    def _project(self, x):
        return torch.cat([self.q_proj(x), self.k_proj(x), self.v_proj(x)], dim=-1)

    def _split_heads(self, qkv_act):
        B, L, _ = qkv_act.shape
        q, k, v = qkv_act.split(self.inner, dim=-1)
        sh = lambda t: t.reshape(B, L, self.n_heads, self.head_dim).transpose(1, 2)    # (B,H,L,hd)
        q, k, v = sh(q), sh(k), sh(v)
        return _l2norm(q) * (self.head_dim ** -0.5), _l2norm(k), v

    def _decay_and_step(self, x):
        beta = torch.sigmoid(self.b_proj(x))                                            # (..., H)
        log_alpha = -self.A_log.exp() * F.softplus(self.a_proj(x) + self.dt_bias)       # (..., H), <= 0
        return log_alpha, beta

    def _finish(self, o, x):
        """o: (B,H,L,hd) memory read-out -> per-head norm -> output gate -> output projection."""
        B, H, L, hd = o.shape
        o = self.out_norm(o.transpose(1, 2))                                            # (B,L,H,hd)
        gate = F.silu(self.g_proj(x)).reshape(B, L, H, hd)
        return self.o_proj((o * gate).reshape(B, L, H * hd))

    def forward(self, x, key_padding_mask=None, return_state=False, use_recurrent=False):
        """x: (B,L,D). key_padding_mask (optional, (B,L) bool, True = real token): LEFT-padding positions are made 'do nothing' tokens
        (zero input, beta = 0, alpha = 1) so the state stays exactly zero until the first real token, whatever the padding."""
        dtype_in = x.dtype
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, L, _ = x.shape
            if key_padding_mask is not None:
                x = x * key_padding_mask.unsqueeze(-1).to(x.dtype)
            qkv = self._project(x)                                                      # (B,L,3*inner) pre-conv
            conv_out = self.conv(F.pad(qkv.transpose(1, 2), (self.conv_width - 1, 0)))  # causal: left zero-pad, output length L
            q, k, v = self._split_heads(F.silu(conv_out.transpose(1, 2)))
            log_alpha, beta = self._decay_and_step(x)                                   # (B,L,H)
            if key_padding_mask is not None:
                m = key_padding_mask.unsqueeze(-1).to(x.dtype)
                log_alpha, beta = log_alpha * m, beta * m
            log_alpha, beta = log_alpha.transpose(1, 2), beta.transpose(1, 2)           # (B,H,L)
            if use_recurrent:
                o, S = gdn_recurrent(q, k, v, log_alpha, beta)
            else:
                o, S = gdn_chunkwise(q, k, v, log_alpha, beta, chunk=self.chunk)
            out = self._finish(o, x)
            if return_state:
                keep = self.conv_width - 1                                              # last pre-conv rows = the decode conv buffer
                buf = qkv[:, -keep:]
                if buf.shape[1] < keep:
                    buf = F.pad(buf, (0, 0, keep - buf.shape[1], 0))
                return out.to(dtype_in), S, buf
        return out.to(dtype_in)

    def step(self, x_t, S, conv_buf):
        """ONE new token per row. x_t: (B,D); S: (B,H,hd,hd) memory; conv_buf: (B,conv_width-1,3*inner) last pre-conv rows.
        Returns (out (B,D), new S, new conv_buf). O(1) work and memory per token."""
        dtype_in = x_t.dtype
        with torch.autocast(device_type=x_t.device.type, enabled=False):
            x = x_t.float()
            qkv_t = self._project(x)                                                    # (B,3*inner)
            window = torch.cat([conv_buf, qkv_t.unsqueeze(1)], dim=1)                   # (B,conv_width,3*inner), oldest first
            w = self.conv.weight.squeeze(1)                                             # (3*inner, conv_width)
            conv_out = (window * w.t().unsqueeze(0)).sum(dim=1)                         # same as the causal conv at this position
            q, k, v = self._split_heads(F.silu(conv_out).unsqueeze(1))                  # (B,H,1,hd)
            q, k, v = q[:, :, 0], k[:, :, 0], v[:, :, 0]                                # (B,H,hd)
            log_alpha, beta = self._decay_and_step(x)                                   # (B,H)
            a, b = log_alpha.exp()[..., None, None], beta[..., None, None]
            Sk = torch.einsum("bhvk,bhk->bhv", S, k)
            S = a * (S - b * Sk.unsqueeze(-1) * k.unsqueeze(-2)) + b * v.unsqueeze(-1) * k.unsqueeze(-2)
            o = torch.einsum("bhvk,bhk->bhv", S, q).unsqueeze(2)                        # (B,H,1,hd)
            out = self._finish(o, x.unsqueeze(1))[:, 0]
        return out.to(dtype_in), S, window[:, 1:]



class CachedDecoderBlock(nn.Module):
    """Pre-norm block. HYBRID_BRANCHES == "both": the attention branch and the Gated-DeltaNet branch read the SAME normalised input; a token-wise
    softmax gate (linear layer, zero-initialised = 50/50) mixes their outputs; the result is added to the residual, then the dense FFN.
    HYBRID_BRANCHES == "attn": plain GQA block (the gate and the GDN branch do not exist)."""

    def __init__(self, d_model, n_heads, d_ff, n_layers, layer_idx=0):
        super().__init__()
        assert NORM_POSITION == "pre" and NORM_TYPE in ("layernorm", "rmsnorm"), "the hybrid block is only written for pre-norm layernorm/rmsnorm"
        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.use_gdn = (HYBRID_BRANCHES == "both")
        if self.use_gdn:
            self.gdn = GatedDeltaNetMixer(d_model)
            self.branch_gate = nn.Linear(d_model, 2)          # token-wise: softmax over {attention, GDN}
            nn.init.zeros_(self.branch_gate.weight)
            nn.init.zeros_(self.branch_gate.bias)
        if layer_idx >= MOE_DENSE_FIRST_LAYERS:
            self.ff = MoEFFN(d_model, MOE_N_ROUTED, MOE_N_SHARED, MOE_TOP_K, MOE_D_EXPERT, MOE_BIAS_RATE)
        else:
            self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)
        self.last_gdn_share = None      # mean softmax weight of the GDN branch in the latest forward (a diagnostic)

    def _mix(self, h, a, g):
        w = torch.softmax(self.branch_gate(h).float(), dim=-1)                        # (..., 2)
        if not self.training:
            self.last_gdn_share = w[..., 1].mean().item() if w.dim() == 3 else None
        return (w[..., 0:1] * a.float() + w[..., 1:2] * g.float()).to(a.dtype)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None, gdn_sink=None):
        h = self.norm1(x)
        a = self.attn(h, key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
        if self.use_gdn:
            if gdn_sink is not None:        # PREFILL: also return the recurrent state and the conv buffer
                g, S, buf = self.gdn(h, key_padding_mask=key_padding_mask, return_state=True)
                gdn_sink.append([S, buf])
            else:
                g = self.gdn(h, key_padding_mask=key_padding_mask)
            a = self._mix(h, a, g)
        x = x + a
        return x + self.ff(self.norm2(x))

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None, gdn_state=None):
        """x_t: (batch, d_model). gdn_state: this layer's [S, conv_buffer] list, updated IN PLACE (O(1) work per token)."""
        h = self.norm1(x_t)
        a = self.attn.decode_step(h, k_cache, v_cache, cache_len, key_mask, rope)
        if self.use_gdn:
            g, S, buf = self.gdn.step(h, gdn_state[0], gdn_state[1])
            gdn_state[0], gdn_state[1] = S, buf
            a = self._mix(h, a, g)
        x_t = x_t + a
        return x_t + self.ff(self.norm2(x_t))


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers, layer_idx=i) for i in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        gdn_sink = []
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope, gdn_sink=gdn_sink)
        cache.length = prefill_len
        cache.canon = gdn_sink if gdn_sink else None      # per layer [S, conv buffer]; the beam search reorders it together with the K/V cache
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope,
                                  gdn_state=(cache.canon[layer_idx] if getattr(cache, "canon", None) else None))
        cache.length = cache_len + 1
        return self.fc_out(h)

## 8. Training chunks, and the held-out Supreme Court chunks used to measure progress

The new corpus is cut into `CONTEXT_LEN`-token chunks exactly as in the base run. Separately, the held-out Supreme Court documents (never trained on) are cut into chunks and a fixed random subset of them is used to measure the language-model loss before, during and after training. This is a stand-in for the hidden test set, and it is the same text for the base model and the this model, so the two numbers are comparable.

In [13]:
num_blocks = max(0, -(-(len(all_word_ids) - CONTEXT_LEN) // CONTEXT_LEN))
_flat = torch.frombuffer(all_word_ids, dtype=torch.int32) if isinstance(all_word_ids, array.array) else torch.tensor(all_word_ids, dtype=torch.int32)
train_data = _flat[:num_blocks * CONTEXT_LEN].view(num_blocks, CONTEXT_LEN).to(torch.long)
del _flat
print(f"New training chunks: {len(train_data):,}  ({len(train_data) * CONTEXT_LEN / 1e6:.0f}M tokens)")

# ---- held-out Supreme Court chunks (fixed subset, same for every model) ----
HELDOUT_EVAL_CHUNKS = 40 if QUICK_TEST else 600
_ho = []
with open(HELDOUT_FILES["supreme_court"], "r", encoding="utf-8") as f:
    for line in f:
        ids = bpe_tokenizer.encode(line.strip())
        for s in range(0, len(ids) - CONTEXT_LEN + 1, CONTEXT_LEN):
            _ho.append(ids[s:s + CONTEXT_LEN])
_sel = random.Random(1234).sample(range(len(_ho)), min(HELDOUT_EVAL_CHUNKS, len(_ho)))
heldout_chunks = torch.tensor([_ho[i] for i in sorted(_sel)], dtype=torch.long)
print(f"Held-out Supreme Court chunks for the loss check: {len(heldout_chunks)} (of {len(_ho)} available)")
del _ho


def heldout_loss(n_chunks=None):
    """Mean next-token cross-entropy (nats/token) of the current model on the held-out Supreme Court chunks, plus perplexity."""
    was_training = model.training
    model.eval()
    chunks = heldout_chunks if n_chunks is None else heldout_chunks[:n_chunks]
    total, count = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(chunks), BATCH_SIZE):
            batch = chunks[s:s + BATCH_SIZE].to(device)
            logits = model(batch[:, :-1])
            total += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1), reduction="sum").item()
            count += batch[:, 1:].numel()
    if was_training:
        model.train()
    loss = total / max(1, count)
    return loss, math.exp(loss)


New training chunks: 89,724  (184M tokens)
Held-out Supreme Court chunks for the loss check: 600 (of 785 available)


## 9. Training on both GPUs

The notebook writes the chunks to disk, writes the training script (`train_ddp.py`, which contains the same model and optimizer code as the cells above) and its settings, tests that the GPUs can communicate, then launches the job and watches it: it prints the job's log as it comes, and pushes the checkpoint file to the bucket whenever a new one has been saved. The job saves a checkpoint every 1000 steps (model + both optimizers + the batch counter). After `MAX_SESSION_HOURS` it saves and stops; **run the notebook again from the top (same settings) to continue**. When the job finishes, the notebook goes on to the held-out loss and the beam-search submission.

In [14]:
# ---- 9a. Save the chunks for the training processes (int32, half the size) ------------------------------------------
import subprocess, sys, json, time, os
WORK = "/kaggle/working"
TRAIN_CHUNKS_PATH, HELDOUT_CHUNKS_PATH = f"{WORK}/train_chunks_{mode_tag}.pt", f"{WORK}/heldout_chunks_{mode_tag}.pt"
torch.save(train_data.to(torch.int32), TRAIN_CHUNKS_PATH)
torch.save(heldout_chunks.to(torch.int32), HELDOUT_CHUNKS_PATH)
n_train_chunks = len(train_data)
del train_data
import gc; gc.collect()
print(f"Saved {n_train_chunks:,} training chunks and {len(heldout_chunks)} held-out chunks.")


Saved 89,724 training chunks and 600 held-out chunks.


In [15]:
%%writefile /kaggle/working/train_ddp.py
# train_ddp.py -- trains the MoE (Muon + final LayerNorm) on ONE OR TWO GPUs with DistributedDataParallel.
# Written by the notebook (%%writefile) and started with torchrun (one process per GPU). All settings come from the JSON file in argv[1].
# Rank 0 does the logging, the held-out evaluation and the checkpoint writing; the notebook pushes the checkpoint file to the bucket.
import os, sys, json, time, math, datetime, traceback
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist

CFG = json.load(open(sys.argv[1]))
RANK, WORLD, LOCAL = int(os.environ.get("RANK", 0)), int(os.environ.get("WORLD_SIZE", 1)), int(os.environ.get("LOCAL_RANK", 0))
USE_CUDA = torch.cuda.is_available()
device = "cuda" if USE_CUDA else "cpu"          # "cuda" = this process's own GPU (set_device below)
if USE_CUDA:
    torch.cuda.set_device(LOCAL)
if WORLD > 1:
    dist.init_process_group("nccl" if USE_CUDA else "gloo", timeout=datetime.timedelta(minutes=90))
CONTEXT_LEN = CFG["context_len"]
NORM_TYPE, NORM_POSITION = "layernorm", "pre"
HYBRID_BRANCHES = CFG["hybrid_branches"]
USE_NATIVE_GQA = False


def say(*a):
    if RANK == 0:
        print(time.strftime("%H:%M:%S"), *a, flush=True)


# ======================================================================================
# Model + optimizer code: identical to the notebook's cells
# ======================================================================================
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

import math
NS_DTYPE = torch.float16 if device == "cuda" else torch.float32     # Newton-Schulz in fp16 uses the T4's tensor cores; the matrices are normalised so fp16 range is fine


def zeropower_batched(G, steps=5, eps=1e-7):
    """Newton-Schulz orthogonalisation of a BATCH of same-shaped matrices G: (B, m, n) (Keller Jordan's quintic coefficients)."""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.to(NS_DTYPE)
    tall = X.size(-2) > X.size(-1)
    if tall:
        X = X.transpose(-2, -1)
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + eps)
    for _ in range(steps):
        A = X @ X.transpose(-2, -1)
        X = a * X + (b * A + c * (A @ A)) @ X
    if tall:
        X = X.transpose(-2, -1)
    return X


class Muon(torch.optim.Optimizer):
    """Muon for hidden weight matrices: momentum (Nesterov) -> orthogonalise -> scale -> decoupled weight decay.
    The update is scaled by 0.2 * sqrt(max(rows, cols)) so its RMS matches AdamW's (Moonlight, Liu et al. 2025);
    that lets Muon reuse AdamW's learning rate and weight decay. Same-shaped matrices are orthogonalised together in one batched call."""

    def __init__(self, params, lr, momentum=0.95, weight_decay=0.1, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr, mu, wd = group["lr"], group["momentum"], group["weight_decay"]
            by_shape = {}
            for p in group["params"]:
                if p.grad is None:
                    continue
                st = self.state[p]
                if "momentum_buffer" not in st:
                    st["momentum_buffer"] = torch.zeros_like(p)
                buf = st["momentum_buffer"]
                buf.mul_(mu).add_(p.grad)
                upd = p.grad.add(buf, alpha=mu)                      # Nesterov
                by_shape.setdefault(tuple(p.shape), []).append((p, upd))
            for shape, items in by_shape.items():
                O = zeropower_batched(torch.stack([u for _, u in items]), steps=group["ns_steps"])
                scale = 0.2 * math.sqrt(max(shape))
                for (p, _), o in zip(items, O):
                    if wd:
                        p.mul_(1 - lr * wd)
                    p.add_(o.to(p.dtype), alpha=-lr * scale)


class MultiOptimizer:
    """Lets the loop treat several optimizers as one."""

    def __init__(self, opts):
        self.opts = opts

    @property
    def param_groups(self):
        return [g for o in self.opts for g in o.param_groups]

    def zero_grad(self):
        for o in self.opts:
            o.zero_grad(set_to_none=True)

    def step(self):
        for o in self.opts:
            o.step()

    def state_dict(self):
        return [o.state_dict() for o in self.opts]

    def load_state_dict(self, states):
        for o, s in zip(self.opts, states):
            o.load_state_dict(s)


def is_muon_matrix(name, p):
    """Hidden 2-D weight matrices (attention projections, expert / FFN matrices). NOT the embedding table, the output head, the router or norms."""
    return p.ndim == 2 and name.startswith("blocks.") and "router" not in name and "norm" not in name


def build_optimizer(model, v, peak_lr):
    kind = v["optimizer"]
    betas = tuple(v.get("betas", (0.9, 0.95)))
    wd = v.get("weight_decay", 0.1)
    named = list(model.named_parameters())
    if kind == "muon":
        muon_params = [p for n, p in named if is_muon_matrix(n, p)]
        rest = [(n, p) for n, p in named if not is_muon_matrix(n, p)]
    else:
        muon_params, rest = [], named
    if v.get("decay_all", False):
        decay = [p for _, p in rest]; no_decay = []
    else:
        decay = [p for n, p in rest if p.ndim >= 2 and "token_emb" not in n and "router" not in n]
        no_decay = [p for n, p in rest if not (p.ndim >= 2 and "token_emb" not in n and "router" not in n)]
    groups = [dict(params=decay, weight_decay=wd)]
    if no_decay:
        groups.append(dict(params=no_decay, weight_decay=0.0))
    adam = torch.optim.AdamW(groups, lr=peak_lr, betas=betas, fused=(device == "cuda"))
    if kind == "muon":
        mu = Muon(muon_params, lr=peak_lr, momentum=v.get("muon_momentum", 0.95), weight_decay=wd, ns_steps=v.get("ns_steps", 5))
        return MultiOptimizer([mu, adam]), dict(muon_tensors=len(muon_params), muon_params=sum(p.numel() for p in muon_params),
                                                adamw_params=sum(p.numel() for g in groups for p in g["params"]))
    return MultiOptimizer([adam]), dict(muon_tensors=0, muon_params=0, adamw_params=sum(p.numel() for g in groups for p in g["params"]))




def build_optimizer(model, v, peak_lr):
    """v: a variant dict (see VARIANT in Settings). Returns (optimizer, info)."""
    kind = v["optimizer"]
    betas = tuple(v.get("betas", (0.9, 0.95)))
    wd = v.get("weight_decay", 0.1)
    named = list(model.named_parameters())
    if kind == "muon":
        muon_params = [p for n, p in named if is_muon_matrix(n, p)]
        rest = [(n, p) for n, p in named if not is_muon_matrix(n, p)]
    else:
        muon_params, rest = [], named
    decay = [p for n, p in rest if p.ndim >= 2 and "token_emb" not in n and "router" not in n]
    no_decay = [p for n, p in rest if not (p.ndim >= 2 and "token_emb" not in n and "router" not in n)]
    groups = [dict(params=decay, weight_decay=wd)]
    if no_decay:
        groups.append(dict(params=no_decay, weight_decay=0.0))
    adam = torch.optim.AdamW(groups, lr=peak_lr, betas=betas, fused=(device == "cuda"))
    if kind == "muon":
        mu = Muon(muon_params, lr=peak_lr, momentum=v.get("muon_momentum", 0.95), weight_decay=wd, ns_steps=v.get("ns_steps", 5))
        return MultiOptimizer([mu, adam]), dict(muon_tensors=len(muon_params), muon_params=sum(p.numel() for p in muon_params),
                                                adamw_params=sum(p.numel() for g in groups for p in g["params"]))
    return MultiOptimizer([adam]), dict(muon_tensors=0, muon_params=0, adamw_params=sum(p.numel() for g in groups for p in g["params"]))


# ---- Settings specific to this notebook ------------------------------
# (GDN_* sizes are defined with the Gated DeltaNet code below)
# ---- Mixture-of-Experts FFN (DeepSeekMoE style) -------------------------------------------------------------
MOE_N_ROUTED = 16          # routed experts per MoE layer (fine-grained: each is only 1/4 the width of the dense FFN)
MOE_N_SHARED = 1           # shared experts: ALWAYS applied to every token (they hold the common knowledge)
MOE_TOP_K = 3              # routed experts used per token. Active FFN width = (1 shared + 3 routed) x 768 = 3072 = the dense FFN's width,
                           # so the compute per token matches the dense GQA run; the TOTAL FFN capacity is 17 x 768 = 4.25x larger.
MOE_D_EXPERT = 768         # hidden width of one expert
MOE_DENSE_FIRST_LAYERS = 99 # >= number of layers: NO layer uses the MoE FFN here (all dense). (The MoE code below is kept only so that the shared helper functions exist.)
MOE_BIAS_RATE = 0.001      # aux-loss-free load balancing (DeepSeek-V3): per step, a per-expert selection bias moves by this much
                           # towards the under-used experts. The bias only affects WHICH experts are chosen, never the output weights.

GQA_GROUP_SIZE = 4      # query heads that share one key/value head. 4 is exactly
                        # what ATTENTION_TYPE = "gqa" meant in the original notebooks.
CACHE_HEADROOM = 512    # slots reserved inside CONTEXT_LEN for GENERATED tokens.
                        # The prompt is cut to its last CONTEXT_LEN - CACHE_HEADROOM
                        # tokens; if generation fills the cache, the window is
                        # re-prefilled (see the markdown above for why). With
                        # CONTEXT_LEN = 2048 that leaves room for a 1536-token prompt
                        # (a 512-word legal prompt is roughly 700-1000 tokens) AND
                        # ~512 generated tokens, so in practice nothing is cut.
USE_NATIVE_GQA = False  # decided automatically by a micro-benchmark in the next
                        # section (True = pass enable_gqa=True to SDPA instead of
                        # expanding K/V with repeat_interleave)
POSITIONAL_ENCODING = "rope"   # "rope" | "learned"
                        # rope    = rotary embeddings: Q and K are rotated by a
                        #           position-dependent angle inside attention, so
                        #           attention scores depend on RELATIVE distance
                        #           (RoFormer, Su et al. 2021). No position table.
                        # learned = a trainable CONTEXT_LEN-row position table added to
                        #           the token embeddings (GPT-2 style, what the original
                        #           decoder-only notebook used).
ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim)
assert POSITIONAL_ENCODING in ("rope", "learned"), "POSITIONAL_ENCODING must be 'rope' or 'learned'"


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


class ExpertMLP(nn.Module):
    """One expert: the same Linear -> GELU -> Linear as the dense FFN, just narrower."""

    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.up = nn.Linear(d_model, d_hidden)
        self.down = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class MoEFFN(nn.Module):
    """DeepSeekMoE feed-forward layer, written from scratch.

        y = sum_s Shared_s(x)  +  sum_{e in TopK(x)} g_e(x) * Routed_e(x)

    * router: affinity s_e = sigmoid(w_e . x) for every routed expert e (DeepSeek-V3 uses sigmoid, not softmax)
    * selection: the top-k experts by (s_e + bias_e); the bias is a NON-learned buffer used only for choosing
    * gates: the selected s_e, renormalised to sum to 1 (so the routed part has a fixed scale)
    * load balancing WITHOUT an auxiliary loss: in training, after each forward pass, bias_e moves by +-MOE_BIAS_RATE
      towards the under-loaded experts (overloaded: down, underloaded: up). Nothing is dropped (no capacity limit).
    Every token is processed independently, so the KV-cache decode path needs no change."""

    def __init__(self, d_model, n_routed, n_shared, top_k, d_expert, bias_rate):
        super().__init__()
        assert 1 <= top_k <= n_routed
        self.n_routed, self.top_k, self.bias_rate = n_routed, top_k, bias_rate
        self.router = nn.Linear(d_model, n_routed, bias=False)
        nn.init.normal_(self.router.weight, std=0.02)
        self.shared = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_shared)])
        self.experts = nn.ModuleList([ExpertMLP(d_model, d_expert) for _ in range(n_routed)])
        self.register_buffer("select_bias", torch.zeros(n_routed))      # saved in the checkpoint (it is part of the trained router)
        self.last_load = None                                           # tokens routed to each expert in the latest training step

    def forward(self, x):
        shape = x.shape
        flat = x.reshape(-1, shape[-1])                                  # (N, d): the router does not care about batch/position
        affinity = torch.sigmoid(self.router(flat).float())              # (N, E), fp32 even under autocast
        _, chosen = (affinity + self.select_bias).topk(self.top_k, dim=-1)       # (N, k): selection uses the bias ...
        gate = affinity.gather(1, chosen)                                # ... but the gate values do not
        gate = gate / gate.sum(dim=-1, keepdim=True)
        out = torch.zeros_like(flat)
        for expert in self.shared:
            out = out + expert(flat)
        for e, expert in enumerate(self.experts):
            token_idx, slot = (chosen == e).nonzero(as_tuple=True)       # the tokens that picked expert e, and in which of their k slots
            if token_idx.numel() == 0:
                continue
            y = expert(flat[token_idx]) * gate[token_idx, slot].unsqueeze(-1).to(flat.dtype)
            out.index_add_(0, token_idx, y.to(out.dtype))
        if self.training:
            with torch.no_grad():
                load = torch.bincount(chosen.reshape(-1), minlength=self.n_routed).float()
                self.last_load = load
                self.select_bias += self.bias_rate * torch.sign(load.mean() - load)
        return out.reshape(shape)


def moe_load_stats(model):
    """[(layer index, busiest expert's load / the average (1.0 = perfectly balanced), number of experts with < 10% of an even share)]
    for every MoE layer, from the latest training step."""
    rows = []
    for i, blk in enumerate(model.blocks):
        if isinstance(blk.ff, MoEFFN) and blk.ff.last_load is not None:
            l = blk.ff.last_load.cpu()
            rows.append((i, round((l.max() / l.mean()).item(), 3), int((l < 0.1 * l.mean()).sum().item())))
    return rows


def moe_load_report(model):
    rows = moe_load_stats(model)
    if not rows:
        print("MoE load report: no training step has run in this session yet.")
        return
    print("MoE load balance (max expert load / mean, experts with <10% of an even share):")
    print("  " + "   ".join(f"layer {i}: {r:.2f}x / {d}" for i, r, d in rows))


def count_active_params(model):
    """Parameters used per token: everything except the routed experts that are NOT selected (top_k of n_routed are used)."""
    total = sum(p.numel() for p in model.parameters())
    inactive = 0
    for blk in model.blocks:
        if isinstance(blk.ff, MoEFFN):
            per_expert = sum(p.numel() for p in blk.ff.experts[0].parameters())
            inactive += per_expert * (blk.ff.n_routed - blk.ff.top_k)
    return total, total - inactive


import math
GDN_HEADS = 4          # heads per Gated DeltaNet layer
GDN_HEAD_DIM = 128     # key dim = value dim per head (inner width = 4 x 128 = 512)
GDN_CONV = 4           # width of the short causal depthwise convolution over q, k, v
GDN_CHUNK = 64         # tokens per chunk in the chunkwise form


def _l2norm(x, eps=1e-6):
    return x / (x.norm(dim=-1, keepdim=True) + eps)


def gdn_recurrent(q, k, v, log_alpha, beta, state=None):
    """REFERENCE (slow, one token at a time). q,k: (B,H,L,dk); v: (B,H,L,dv); log_alpha, beta: (B,H,L); state: (B,H,dv,dk) or None.
    Returns outputs (B,H,L,dv) and the final state."""
    B, H, L, dk = q.shape
    dv = v.shape[-1]
    S = q.new_zeros(B, H, dv, dk) if state is None else state
    outs = []
    for t in range(L):
        kt, vt, qt = k[:, :, t], v[:, :, t], q[:, :, t]
        a = log_alpha[:, :, t].exp()[..., None, None]
        b = beta[:, :, t][..., None, None]
        Sk = torch.einsum("bhvk,bhk->bhv", S, kt)
        S = a * (S - b * Sk.unsqueeze(-1) * kt.unsqueeze(-2)) + b * vt.unsqueeze(-1) * kt.unsqueeze(-2)
        outs.append(torch.einsum("bhvk,bhk->bhv", S, qt))
    return torch.stack(outs, dim=2), S


def gdn_chunkwise(q, k, v, log_alpha, beta, state=None, chunk=64):
    """Same function as gdn_recurrent, computed `chunk` tokens at a time.
    Inside a chunk (c_i = cumulative log-decay, gamma_i = exp(c_i)) the state is  S_i = gamma_i S0 + sum_{j<=i} (gamma_i/gamma_j) d_j k_j^T,
    where the 'corrected values' d_j solve a small lower-triangular system:  (I + A) D = beta*V - (beta*gamma*K) S0^T ,
    A_ij = beta_i (gamma_i/gamma_j) k_i.k_j for j < i.   With T = (I+A)^-1:  U = T (beta V),  W = T (beta gamma K),  D = U - W S0^T.
    Then   O = diag(gamma) Q S0^T + ((Q K^T) * Lambda) D ,   S_end = gamma_C S0 + D^T diag(gamma_C/gamma) K.
    Decay ratios are built as exp(c_i - c_j) AFTER masking j > i (never a division of cumulative products), so nothing over/underflows."""
    B, H, L, dk = q.shape
    dv = v.shape[-1]
    pad = (-L) % chunk
    if pad:   # right-pad with 'do nothing' tokens (k = v = q = 0, beta = 0, alpha = 1): the state is untouched by them
        q, k, v = (F.pad(x, (0, 0, 0, pad)) for x in (q, k, v))
        log_alpha, beta = (F.pad(x, (0, pad)) for x in (log_alpha, beta))
    N = (L + pad) // chunk
    q, k, v = (x.reshape(B, H, N, chunk, x.shape[-1]) for x in (q, k, v))
    la, be = (x.reshape(B, H, N, chunk) for x in (log_alpha, beta))
    c = la.cumsum(dim=-1)                                                         # (B,H,N,C) inclusive cumulative log-decay
    on_or_below = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))
    strictly_below = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device), diagonal=-1)
    lam = (c.unsqueeze(-1) - c.unsqueeze(-2)).masked_fill(~on_or_below, float("-inf")).exp()   # gamma_i/gamma_j for j<=i, else 0
    A = (be.unsqueeze(-1) * (k @ k.transpose(-1, -2)) * lam).masked_fill(~strictly_below, 0.0)
    eye = torch.eye(chunk, dtype=q.dtype, device=q.device)
    T = torch.linalg.solve_triangular(eye + A, eye.expand_as(A), upper=False)      # (I + A)^-1
    u = T @ (be.unsqueeze(-1) * v)                                                 # (B,H,N,C,dv)
    w = T @ ((be * c.exp()).unsqueeze(-1) * k)                                     # (B,H,N,C,dk)
    qk = (q @ k.transpose(-1, -2)) * lam                                           # (B,H,N,C,C)
    q_g = c.exp().unsqueeze(-1) * q
    c_last = c[..., -1]                                                            # (B,H,N)
    k_dec = (c_last.unsqueeze(-1) - c).exp().unsqueeze(-1) * k
    S = q.new_zeros(B, H, dv, dk) if state is None else state
    outs = []
    for n in range(N):
        St = S.transpose(-1, -2)                                                   # (B,H,dk,dv)
        d = u[:, :, n] - w[:, :, n] @ St                                           # corrected values (B,H,C,dv)
        outs.append(q_g[:, :, n] @ St + qk[:, :, n] @ d)
        S = c_last[:, :, n].exp()[..., None, None] * S + d.transpose(-1, -2) @ k_dec[:, :, n]
    o = torch.stack(outs, dim=2).reshape(B, H, N * chunk, dv)[:, :, :L]
    return o, S


class GatedDeltaNetMixer(nn.Module):
    """The sequence-mixing part of a Gated DeltaNet layer (the block wraps it with the norm, residual and MLP).
    x -> q,k,v projections -> short causal depthwise conv + SiLU -> L2-normalised q,k -> delta-rule memory (gate alpha, step beta)
      -> per-head RMSNorm -> times SiLU(output gate) -> output projection.   Always computed in fp32 (the recurrence is the numerically
    sensitive part; it is a small share of the compute)."""

    def __init__(self, d_model, n_heads=GDN_HEADS, head_dim=GDN_HEAD_DIM, conv_width=GDN_CONV, chunk=GDN_CHUNK):
        super().__init__()
        self.n_heads, self.head_dim, self.conv_width, self.chunk = n_heads, head_dim, conv_width, chunk
        inner = n_heads * head_dim
        self.inner = inner
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.g_proj = nn.Linear(d_model, inner, bias=False)          # output gate
        self.o_proj = nn.Linear(inner, d_model, bias=False)
        self.a_proj = nn.Linear(d_model, n_heads, bias=False)        # -> decay alpha_t (per head)
        self.b_proj = nn.Linear(d_model, n_heads, bias=False)        # -> step size beta_t (per head)
        self.conv = nn.Conv1d(3 * inner, 3 * inner, conv_width, groups=3 * inner, bias=False)   # depthwise, applied to [q;k;v]
        self.out_norm = RMSNorm(head_dim)
        # decay parameterisation (Mamba-2 style): alpha = exp(-exp(A_log) * softplus(a_proj(x) + dt_bias)), A in [1,16], dt in [1e-3, 1e-1]
        self.A_log = nn.Parameter(torch.log(torch.empty(n_heads).uniform_(1.0, 16.0)))
        dt = torch.exp(torch.rand(n_heads) * (math.log(0.1) - math.log(0.001)) + math.log(0.001))
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))        # inverse softplus of dt

    def _project(self, x):
        return torch.cat([self.q_proj(x), self.k_proj(x), self.v_proj(x)], dim=-1)

    def _split_heads(self, qkv_act):
        B, L, _ = qkv_act.shape
        q, k, v = qkv_act.split(self.inner, dim=-1)
        sh = lambda t: t.reshape(B, L, self.n_heads, self.head_dim).transpose(1, 2)    # (B,H,L,hd)
        q, k, v = sh(q), sh(k), sh(v)
        return _l2norm(q) * (self.head_dim ** -0.5), _l2norm(k), v

    def _decay_and_step(self, x):
        beta = torch.sigmoid(self.b_proj(x))                                            # (..., H)
        log_alpha = -self.A_log.exp() * F.softplus(self.a_proj(x) + self.dt_bias)       # (..., H), <= 0
        return log_alpha, beta

    def _finish(self, o, x):
        """o: (B,H,L,hd) memory read-out -> per-head norm -> output gate -> output projection."""
        B, H, L, hd = o.shape
        o = self.out_norm(o.transpose(1, 2))                                            # (B,L,H,hd)
        gate = F.silu(self.g_proj(x)).reshape(B, L, H, hd)
        return self.o_proj((o * gate).reshape(B, L, H * hd))

    def forward(self, x, key_padding_mask=None, return_state=False, use_recurrent=False):
        """x: (B,L,D). key_padding_mask (optional, (B,L) bool, True = real token): LEFT-padding positions are made 'do nothing' tokens
        (zero input, beta = 0, alpha = 1) so the state stays exactly zero until the first real token, whatever the padding."""
        dtype_in = x.dtype
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            B, L, _ = x.shape
            if key_padding_mask is not None:
                x = x * key_padding_mask.unsqueeze(-1).to(x.dtype)
            qkv = self._project(x)                                                      # (B,L,3*inner) pre-conv
            conv_out = self.conv(F.pad(qkv.transpose(1, 2), (self.conv_width - 1, 0)))  # causal: left zero-pad, output length L
            q, k, v = self._split_heads(F.silu(conv_out.transpose(1, 2)))
            log_alpha, beta = self._decay_and_step(x)                                   # (B,L,H)
            if key_padding_mask is not None:
                m = key_padding_mask.unsqueeze(-1).to(x.dtype)
                log_alpha, beta = log_alpha * m, beta * m
            log_alpha, beta = log_alpha.transpose(1, 2), beta.transpose(1, 2)           # (B,H,L)
            if use_recurrent:
                o, S = gdn_recurrent(q, k, v, log_alpha, beta)
            else:
                o, S = gdn_chunkwise(q, k, v, log_alpha, beta, chunk=self.chunk)
            out = self._finish(o, x)
            if return_state:
                keep = self.conv_width - 1                                              # last pre-conv rows = the decode conv buffer
                buf = qkv[:, -keep:]
                if buf.shape[1] < keep:
                    buf = F.pad(buf, (0, 0, keep - buf.shape[1], 0))
                return out.to(dtype_in), S, buf
        return out.to(dtype_in)

    def step(self, x_t, S, conv_buf):
        """ONE new token per row. x_t: (B,D); S: (B,H,hd,hd) memory; conv_buf: (B,conv_width-1,3*inner) last pre-conv rows.
        Returns (out (B,D), new S, new conv_buf). O(1) work and memory per token."""
        dtype_in = x_t.dtype
        with torch.autocast(device_type=x_t.device.type, enabled=False):
            x = x_t.float()
            qkv_t = self._project(x)                                                    # (B,3*inner)
            window = torch.cat([conv_buf, qkv_t.unsqueeze(1)], dim=1)                   # (B,conv_width,3*inner), oldest first
            w = self.conv.weight.squeeze(1)                                             # (3*inner, conv_width)
            conv_out = (window * w.t().unsqueeze(0)).sum(dim=1)                         # same as the causal conv at this position
            q, k, v = self._split_heads(F.silu(conv_out).unsqueeze(1))                  # (B,H,1,hd)
            q, k, v = q[:, :, 0], k[:, :, 0], v[:, :, 0]                                # (B,H,hd)
            log_alpha, beta = self._decay_and_step(x)                                   # (B,H)
            a, b = log_alpha.exp()[..., None, None], beta[..., None, None]
            Sk = torch.einsum("bhvk,bhk->bhv", S, k)
            S = a * (S - b * Sk.unsqueeze(-1) * k.unsqueeze(-2)) + b * v.unsqueeze(-1) * k.unsqueeze(-2)
            o = torch.einsum("bhvk,bhk->bhv", S, q).unsqueeze(2)                        # (B,H,1,hd)
            out = self._finish(o, x.unsqueeze(1))[:, 0]
        return out.to(dtype_in), S, window[:, 1:]



class CachedDecoderBlock(nn.Module):
    """Pre-norm block. HYBRID_BRANCHES == "both": the attention branch and the Gated-DeltaNet branch read the SAME normalised input; a token-wise
    softmax gate (linear layer, zero-initialised = 50/50) mixes their outputs; the result is added to the residual, then the dense FFN.
    HYBRID_BRANCHES == "attn": plain GQA block (the gate and the GDN branch do not exist)."""

    def __init__(self, d_model, n_heads, d_ff, n_layers, layer_idx=0):
        super().__init__()
        assert NORM_POSITION == "pre" and NORM_TYPE in ("layernorm", "rmsnorm"), "the hybrid block is only written for pre-norm layernorm/rmsnorm"
        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.use_gdn = (HYBRID_BRANCHES == "both")
        if self.use_gdn:
            self.gdn = GatedDeltaNetMixer(d_model)
            self.branch_gate = nn.Linear(d_model, 2)          # token-wise: softmax over {attention, GDN}
            nn.init.zeros_(self.branch_gate.weight)
            nn.init.zeros_(self.branch_gate.bias)
        if layer_idx >= MOE_DENSE_FIRST_LAYERS:
            self.ff = MoEFFN(d_model, MOE_N_ROUTED, MOE_N_SHARED, MOE_TOP_K, MOE_D_EXPERT, MOE_BIAS_RATE)
        else:
            self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)
        self.last_gdn_share = None      # mean softmax weight of the GDN branch in the latest forward (a diagnostic)

    def _mix(self, h, a, g):
        w = torch.softmax(self.branch_gate(h).float(), dim=-1)                        # (..., 2)
        if not self.training:
            self.last_gdn_share = w[..., 1].mean().item() if w.dim() == 3 else None
        return (w[..., 0:1] * a.float() + w[..., 1:2] * g.float()).to(a.dtype)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None, gdn_sink=None):
        h = self.norm1(x)
        a = self.attn(h, key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
        if self.use_gdn:
            if gdn_sink is not None:        # PREFILL: also return the recurrent state and the conv buffer
                g, S, buf = self.gdn(h, key_padding_mask=key_padding_mask, return_state=True)
                gdn_sink.append([S, buf])
            else:
                g = self.gdn(h, key_padding_mask=key_padding_mask)
            a = self._mix(h, a, g)
        x = x + a
        return x + self.ff(self.norm2(x))

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None, gdn_state=None):
        """x_t: (batch, d_model). gdn_state: this layer's [S, conv_buffer] list, updated IN PLACE (O(1) work per token)."""
        h = self.norm1(x_t)
        a = self.attn.decode_step(h, k_cache, v_cache, cache_len, key_mask, rope)
        if self.use_gdn:
            g, S, buf = self.gdn.step(h, gdn_state[0], gdn_state[1])
            gdn_state[0], gdn_state[1] = S, buf
            a = self._mix(h, a, g)
        x_t = x_t + a
        return x_t + self.ff(self.norm2(x_t))


class CachedDecoderModel(nn.Module):
    """GPT-style decoder, GQA attention, KV-cache aware. Positions come from
    POSITIONAL_ENCODING: "learned" (pos_emb table added to the token
    embeddings) or "rope" (rotations inside attention, no table)."""

    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, context_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.use_rope = (POSITIONAL_ENCODING == "rope")
        if self.use_rope:
            cos, sin = build_rope_cache(context_len, d_model // n_heads, ROPE_BASE)
            # tables, not parameters; persistent=False keeps them out of state_dict
            # (one pair shared by every layer -- they depend only on position)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.pos_emb = nn.Embedding(context_len, d_model)
        self.blocks = nn.ModuleList([
            CachedDecoderBlock(d_model, n_heads, d_ff, n_layers, layer_idx=i) for i in range(n_layers)
        ])
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _embed(self, token_ids, position_ids):
        """Token embedding (+ learned position embedding when not using RoPE)."""
        h = self.token_emb(token_ids)
        if not self.use_rope:
            h = h + self.pos_emb(position_ids)
        return h

    def forward(self, x, key_padding_mask=None, position_ids=None, last_position_only=False):
        """Training / full-recompute forward. x: (batch, seq) token ids.
        position_ids (optional, (batch, seq)) lets padded batches give each
        row's first real token position 0. last_position_only=True skips the
        big vocab projection for every position but the last (used by the
        uncached reference generator)."""
        batch_size, seq_len = x.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device).unsqueeze(0)
        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]   # (batch or 1, seq, head_dim)
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))                            # add the heads axis
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        if last_position_only:
            h = h[:, -1, :]
        return self.fc_out(h)

    def prefill(self, x, pad_lens, cache):
        """PREFILL. x: (batch, P) token ids, LEFT-padded; pad_lens: list of
        ints (pad count per row). Fills cache positions [0:P] for every layer,
        sets cache.length = P, and returns ONLY the last position's logits
        (batch, vocab) -- the one thing generation needs."""
        batch_size, prefill_len = x.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=x.device)
        arange = torch.arange(prefill_len, device=x.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)   # first REAL token -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None

        h = self._embed(x, position_ids)
        rope = None
        if self.use_rope:
            cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]
            rope = (cos.unsqueeze(1), sin.unsqueeze(1))
        gdn_sink = []
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope, gdn_sink=gdn_sink)
        cache.length = prefill_len
        cache.canon = gdn_sink if gdn_sink else None      # per layer [S, conv buffer]; the beam search reorders it together with the K/V cache
        return self.fc_out(h[:, -1, :])

    def decode_step(self, token_ids, pad_t, has_padding, cache):
        """DECODE. token_ids: (batch,) -- ONE new token per row. pad_t: (batch,)
        long tensor of pad counts. Appends to the cache, returns logits
        (batch, vocab) for the NEXT token. O(1) new work per step (plus one
        read of the cache)."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)              # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=token_ids.device).unsqueeze(0) >= pad_t.unsqueeze(1)

        h = self._embed(token_ids, position_ids)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids]) if self.use_rope else None   # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope,
                                  gdn_state=(cache.canon[layer_idx] if getattr(cache, "canon", None) else None))
        cache.length = cache_len + 1
        return self.fc_out(h)
for _k, _v in CFG.get("model_overrides", {}).items():       # only used by the CPU test (tiny dims)
    globals()[_k] = _v
USE_NATIVE_GQA = False

# ======================================================================================
# Setup
# ======================================================================================
train_data = torch.load(CFG["train_chunks_path"])        # (n, CONTEXT_LEN) int32
held = torch.load(CFG["heldout_chunks_path"])
B = CFG["per_gpu_batch"]
G = B * WORLD                                             # sequences per optimizer step, over all GPUs
n_batches = math.ceil(len(train_data) / G)
total = n_batches if CFG.get("max_steps") is None else min(CFG["max_steps"], n_batches)
V = CFG["variant"]
dims = CFG["model_dims"]
CKPT = CFG["checkpoint_path"]

torch.manual_seed(CFG["init_seed"])
raw = CachedDecoderModel(dims["vocab"], d_model=dims["d_model"], n_heads=dims["n_heads"], n_layers=dims["n_layers"], d_ff=dims["d_ff"], context_len=CONTEXT_LEN)
if CFG["final_norm"]:
    raw.fc_out = nn.Sequential(make_norm(dims["d_model"], NORM_TYPE), raw.fc_out)
raw = raw.to(device).train()
optimizer, opt_info = build_optimizer(raw, V, CFG["peak_lr"])

start_step, loss_sum, started_from = 0, 0.0, "scratch (random initialisation)"
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    for key, now in (("world_size", WORLD), ("per_gpu_batch", B), ("total_steps", total), ("num_train_chunks", len(train_data))):
        if ck.get(key) != now:
            raise RuntimeError(f"checkpoint was written with {key}={ck.get(key)} but this run has {now}: the schedule would no longer line up. "
                               "Use the same number of GPUs / settings as the first session, or delete the checkpoint to start over.")
    raw.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_step, loss_sum = ck["next_batch"], ck.get("loss_sum", 0.0)
    started_from = f"this run's own checkpoint (batch {start_step})"
    if ck.get("finished"):
        say("This run already finished earlier; nothing to do.")
        sys.exit(0)
    del ck
model = raw
if WORLD > 1:
    model = nn.parallel.DistributedDataParallel(raw, device_ids=[LOCAL] if USE_CUDA else None, find_unused_parameters=True)   # an expert that gets no token in a step has no gradient

n_params = sum(p.numel() for p in raw.parameters())
say(f"{n_params:,} parameters; {WORLD} GPU(s) x {B} sequences = {G} per step; {total} steps; optimizer {V['optimizer']} {opt_info}; started from {started_from}")
say(f"final_norm={CFG['final_norm']}  peak lr {CFG['peak_lr']}  warm-up {CFG['warmup_steps']}  decay starts at {CFG['decay_start_fraction']:.0%} of the run, down to {CFG['min_lr_fraction']:.0%}")

decay_start = max(CFG["warmup_steps"], int(CFG["decay_start_fraction"] * total))
min_lr = CFG["peak_lr"] * CFG["min_lr_fraction"]


def lr_at(step):
    if step < CFG["warmup_steps"]:
        return CFG["peak_lr"] * (step + 1) / CFG["warmup_steps"]
    if step < decay_start:
        return CFG["peak_lr"]
    prog = min(1.0, (step - decay_start) / max(1, total - decay_start))
    return min_lr + 0.5 * (CFG["peak_lr"] - min_lr) * (1 + math.cos(math.pi * prog))


def heldout_loss(n_chunks=None):
    raw.eval()
    chunks = held if n_chunks is None else held[:n_chunks]
    tot, cnt = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(chunks), B):
            b = chunks[s:s + B].to(device).long()
            logits = raw(b[:, :-1])
            tot += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), b[:, 1:].reshape(-1), reduction="sum").item()
            cnt += b[:, 1:].numel()
    raw.train()
    return tot / max(1, cnt)


def save(next_batch, finished=False):
    state = {"model": raw.state_dict(), "optimizer": optimizer.state_dict(), "epoch": 0, "next_batch": next_batch, "loss_sum": loss_sum,
             "total_steps": total, "finished": finished, "world_size": WORLD, "per_gpu_batch": B, "num_train_chunks": len(train_data)}
    tmp = CKPT + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, CKPT)


def sync_replicas():
    """Insurance: copy rank 0's weights and buffers to every rank (they should be identical already; this removes any tiny numerical drift)."""
    if WORLD > 1:
        with torch.no_grad():
            for t in list(raw.parameters()) + list(raw.buffers()):
                dist.broadcast(t.data, 0)


order = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(CFG["data_seed"]))
eval_every, ckpt_every = CFG["eval_every"], CFG["ckpt_every"]
t0, step0 = time.time(), start_step
recent = []
stopped_early = False
model.train()
for step in range(start_step, total):
    batch = train_data[order[step * G + RANK * B: step * G + (RANK + 1) * B]].to(device).long()
    lr = lr_at(step)
    for g in optimizer.param_groups:
        g["lr"] = lr
    logits = model(batch[:, :-1])
    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), batch[:, 1:].reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    if V.get("grad_clip"):
        torch.nn.utils.clip_grad_norm_(raw.parameters(), V["grad_clip"])
    optimizer.step()
    li = loss.item()
    if not math.isfinite(li):
        print(f"[rank {RANK}] loss is {li} at step {step}: stopping", flush=True)
        sys.exit(2)
    loss_sum += li
    recent.append(li)
    if len(recent) > 100:
        recent.pop(0)
    done = step + 1
    if done % 100 == 0:
        rate = (done - step0) / (time.time() - t0)
        say(f"step {done}/{total}  loss(avg last 100, rank 0) {sum(recent) / len(recent):.3f}  lr {lr:.2e}  {rate:.2f} step/s  ~{(total - done) / rate / 3600:.1f} h left")
        if done % 500 == 0:
            if RANK == 0:
                _s = moe_load_stats(raw)
                if _s:
                    say("    MoE load (busiest expert / mean; dead experts) per layer: " + "  ".join(f"L{i}:{r:.2f}x/{d}" for i, r, d in _s))
    if done % CFG["sync_every"] == 0:
        sync_replicas()
    if done % eval_every == 0 and done < total:
        if RANK == 0:
            say(f"held-out Supreme Court loss (first 100 chunks) at step {done}: {heldout_loss(100):.4f}")
        if WORLD > 1:
            dist.barrier()
    if done % ckpt_every == 0 and done < total:
        if RANK == 0:
            save(done)
        if WORLD > 1:
            dist.barrier()
    if done % 20 == 0:
        stop = torch.tensor([1 if (time.time() - t0) / 3600 > CFG["max_session_hours"] else 0], device=device)
        if WORLD > 1:
            dist.broadcast(stop, 0)
        if stop.item() and done < total:
            if RANK == 0:
                save(done)
            say(f"reached max_session_hours={CFG['max_session_hours']} at step {done}/{total}: state saved. Run the notebook again (same settings) to continue.")
            stopped_early = True
            break

if not stopped_early:
    sync_replicas()
    if RANK == 0:
        final = heldout_loss()
        save(total, finished=True)
        say(f"TRAINING COMPLETE. average training loss {loss_sum / max(1, total):.3f}; final held-out loss ({len(held)} chunks) {final:.4f} (ppl {math.exp(final):.2f})")
    if WORLD > 1:
        dist.barrier()
if WORLD > 1:
    dist.destroy_process_group()


Writing /kaggle/working/train_ddp.py


In [16]:
%%writefile /kaggle/working/nccl_check.py
# Quick multi-GPU sanity check run before training: do the two GPUs talk to each other (NCCL all-reduce)? Prints the bandwidth.
import os, time, torch, torch.distributed as dist, datetime
r, w, l = int(os.environ["RANK"]), int(os.environ["WORLD_SIZE"]), int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(l)
dist.init_process_group("nccl", timeout=datetime.timedelta(seconds=120))
x = torch.ones(64 * 1024 * 1024, device="cuda")      # 256 MB
dist.all_reduce(x); torch.cuda.synchronize()
t = time.time()
for _ in range(4):
    dist.all_reduce(x)
torch.cuda.synchronize()
dt = (time.time() - t) / 4
if r == 0:
    print(f"NCCL_OK  all-reduce of 256 MB: {dt * 1000:.0f} ms  (~{0.25 / dt:.1f} GB/s)  value check {x[0].item():.3g}", flush=True)
dist.destroy_process_group()


Writing /kaggle/working/nccl_check.py


In [17]:
# ---- 9b. How many GPUs, and can they talk to each other? -----------------------------------------------------------
def run_nccl_check(n, extra_env=None, timeout=240):
    env = dict(os.environ, **(extra_env or {}))
    try:
        r = subprocess.run([sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={n}", f"{WORK}/nccl_check.py"],
                           capture_output=True, text=True, timeout=timeout, env=env)
        ok = r.returncode == 0 and "NCCL_OK" in r.stdout
        line = [l for l in r.stdout.splitlines() if "NCCL_OK" in l]
        return ok, (line[0] if line else (r.stdout + r.stderr)[-600:])
    except subprocess.TimeoutExpired:
        return False, f"timed out after {timeout}s"

available = torch.cuda.device_count() if device == "cuda" else 0
world = min(available, NUM_GPUS) if NUM_GPUS else max(1, available)
launch_env = {}
if QUICK_TEST and available == 0:
    world = 2                                   # CPU pipeline test: two processes with the gloo backend
    print("No GPU: testing the 2-process pipeline on the CPU (gloo).")
elif world > 1:
    ok, msg = run_nccl_check(world)
    print("NCCL check:", msg)
    if not ok:
        ok, msg = run_nccl_check(world, {"NCCL_P2P_DISABLE": "1"})
        print("NCCL check with NCCL_P2P_DISABLE=1:", msg)
        if ok:
            launch_env["NCCL_P2P_DISABLE"] = "1"
        else:
            print("The GPUs cannot talk to each other -> training on ONE GPU. Note: this makes the global batch "
                  f"{PER_GPU_BATCH} instead of {PER_GPU_BATCH * world}; use the same setting in every session of this run.")
            world = 1
print(f"Training with {world} process(es), {PER_GPU_BATCH} sequences each -> global batch {world * PER_GPU_BATCH}.")


NCCL check: NCCL_OK  all-reduce of 256 MB: 36 ms  (~7.0 GB/s)  value check 32
Training with 2 process(es), 2 sequences each -> global batch 4.


In [18]:
# ---- 9c. Settings file for the job ----------------------------------------------------------------------------------
CFG_PATH = f"{WORK}/train_ddp_cfg_{mode_tag}.json"
cfg = dict(context_len=CONTEXT_LEN, train_chunks_path=TRAIN_CHUNKS_PATH, heldout_chunks_path=HELDOUT_CHUNKS_PATH,
           per_gpu_batch=PER_GPU_BATCH, max_steps=MAX_TRAIN_BATCHES, variant=VARIANT,
           model_dims=dict(vocab=bpe_tokenizer.get_vocab_size(), d_model=768, n_heads=12, n_layers=8, d_ff=3072),
           checkpoint_path=CHECKPOINT_PATH, init_seed=INIT_SEED, final_norm=FINAL_NORM, peak_lr=PEAK_LR, warmup_steps=WARMUP_STEPS,
           decay_start_fraction=DECAY_START_FRACTION, min_lr_fraction=MIN_LR_FRACTION,
           eval_every=30 if QUICK_TEST else 4000, ckpt_every=20 if QUICK_TEST else 1000, sync_every=SYNC_REPLICAS_EVERY,
           max_session_hours=MAX_SESSION_HOURS, data_seed=DATA_ORDER_SEED, model_overrides={}, hybrid_branches=HYBRID_BRANCHES)
json.dump(cfg, open(CFG_PATH, "w"))
log_event("ddp_train_launch", world=world, per_gpu_batch=PER_GPU_BATCH, n_train_chunks=n_train_chunks, peak_lr=PEAK_LR, optimizer=VARIANT["optimizer"], final_norm=FINAL_NORM)


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


In [19]:
# ---- 9d. Launch the job and watch it ----------------------------------------------------------------------------------
try_pull_checkpoint_from_hub(CHECKPOINT_PATH)          # a run started earlier (other session / other account)?
LOG_PATH = f"{WORK}/train_ddp_{mode_tag}.log"
cmd = [sys.executable, f"{WORK}/train_ddp.py", CFG_PATH] if world == 1 else \
      [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={world}", f"{WORK}/train_ddp.py", CFG_PATH]
env = dict(os.environ, **launch_env, PYTHONUNBUFFERED="1", OMP_NUM_THREADS=os.environ.get("OMP_NUM_THREADS", "4"))
logf = open(LOG_PATH, "w")
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env)
print("Started training job (pid %d). Log lines appear below; checkpoints are pushed to the bucket as they are saved." % proc.pid)

def ckpt_mtime():
    return os.path.getmtime(CHECKPOINT_PATH) if os.path.exists(CHECKPOINT_PATH) else None

def push_ckpt():
    try:
        push_checkpoint_to_hub(CHECKPOINT_PATH, quiet=True)
        sync_run_log()
        print(f"[{time.strftime('%H:%M:%S')}] checkpoint pushed to the bucket")
    except Exception as e:
        print(f"(checkpoint push failed: {type(e).__name__} -- will retry)")

last_pos, last_pushed_mtime, last_push_time = 0, ckpt_mtime(), 0.0     # (an old checkpoint from a previous session is already in the bucket)
POLL_SECONDS = 5 if QUICK_TEST else 60
while True:
    time.sleep(POLL_SECONDS)
    logf.flush()
    with open(LOG_PATH, "r") as f:
        f.seek(last_pos); new = f.read(); last_pos = f.tell()
    if new:
        print(new, end="")
    m = ckpt_mtime()
    if m and m != last_pushed_mtime and (time.time() - last_push_time) / 60 >= PUSH_MIN_MINUTES and proc.poll() is None:
        push_ckpt(); last_pushed_mtime, last_push_time = m, time.time()
    if proc.poll() is not None:
        break
logf.flush()
with open(LOG_PATH, "r") as f:
    f.seek(last_pos); print(f.read(), end="")
if proc.returncode != 0:
    tail = "".join(open(LOG_PATH).readlines()[-40:])
    raise RuntimeError(f"The training job exited with code {proc.returncode}. Last log lines:\n{tail}")
push_ckpt()


No matching checkpoint on the Hub yet (FileNotFoundError) -- starting fresh.
Started training job (pid 225). Log lines appear below; checkpoints are pushed to the bucket as they are saved.
05:50:45 126,937,744 parameters; 2 GPU(s) x 2 sequences = 4 per step; 22431 steps; optimizer adamw {'muon_tensors': 0, 'muon_params': 0, 'adamw_params': 126937744}; started from scratch (random initialisation)
05:50:45 final_norm=True  peak lr 0.0003  warm-up 300  decay starts at 0% of the run, down to 10%
[rank1]:[W1005 05:50:46.439296647 reducer.cpp:1502] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later it

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

... (131 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

[14:02:28] checkpoint pushed to the bucket


In [20]:
# ---- 9e. Did the run finish (all steps) or stop at MAX_SESSION_HOURS? ------------------------------------------------
try:
    _ck = torch.load(CHECKPOINT_PATH, map_location="cpu", mmap=True)
except Exception:
    _ck = torch.load(CHECKPOINT_PATH, map_location="cpu")
TRAINING_FINISHED = bool(_ck.get("finished", False))
print(f"Checkpoint is at batch {_ck['next_batch']}/{_ck['total_steps']}; finished = {TRAINING_FINISHED}")
log_event("ddp_train_session_end", next_batch=_ck["next_batch"], total_steps=_ck["total_steps"], finished=TRAINING_FINISHED)
sync_run_log()
if not TRAINING_FINISHED:
    print("Not finished yet -- run the notebook again from the top (same settings) to continue; the cells below are skipped until then.")


Checkpoint is at batch 22431/22431; finished = True


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 10. Build the model (with the final LayerNorm) and load the trained weights

In [21]:
vocab_size = bpe_tokenizer.get_vocab_size()
torch.manual_seed(INIT_SEED)
model = CachedDecoderModel(vocab_size, d_model=768, n_heads=12, n_layers=8,
                           d_ff=3072, context_len=CONTEXT_LEN)
if FINAL_NORM:
    model.fc_out = nn.Sequential(make_norm(768, NORM_TYPE), model.fc_out)     # a LayerNorm before the output head (every code path calls self.fc_out(h), so cached generation needs no change)
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,} ({num_params / 1e6:.1f}M)")
_, active_params = count_active_params(model)
print(f"Parameters per token: {active_params:,} (dense model: all of them are used)")
if not (50_000_000 <= num_params <= 300_000_000):
    print("WARNING: this is outside the required 50M-300M range! "
          "Change d_model / n_layers above.")

# ---- Checkpoint name records the positional-encoding choice ----------------
# A RoPE model has no position table, so its weights cannot load into a
# "learned" model or vice versa, and a model trained at one CONTEXT_LEN should not
# silently resume into a run at another; the filename records both.
print(f"Positional encoding: {POSITIONAL_ENCODING}   context length: {CONTEXT_LEN}")
# the checkpoint name also records the position mode, the context length and how much data
# the tokenizer + model were trained on (a different data size means a different tokenizer,
# so its checkpoint must never be resumed into another run)
DATA_TAG = "160Mw"   # (the checkpoint name is fixed in Settings)
# "a1bpe" = the from-scratch Assignment-1 BPE of Section 5; a checkpoint trained with any other tokenizer has different token ids.
pass   # CHECKPOINT_PATH is set explicitly in Settings

# (The old "reuse the original notebook's checkpoint" option is gone: those checkpoints were trained
# with a different tokenizer, so their token ids would not line up with this notebook's own BPE.)

# ---- KV-cache memory: GQA (this model) vs the MHA equivalent ---------------
_attn0 = model.blocks[0].attn
_bytes_per_token_gqa = 2 * len(model.blocks) * _attn0.n_kv_heads * _attn0.head_dim * 2   # K and V, fp16
_bytes_per_token_mha = 2 * len(model.blocks) * _attn0.n_heads * _attn0.head_dim * 2
print(f"\nKV cache per sequence at {CONTEXT_LEN} tokens (fp16): "
      f"GQA {_bytes_per_token_gqa * CONTEXT_LEN / 1e6:.1f} MB "
      f"vs MHA {_bytes_per_token_mha * CONTEXT_LEN / 1e6:.1f} MB "
      f"({_attn0.n_heads // _attn0.n_kv_heads}x smaller)")


# ---- Which fused-attention kernels can this GPU / dtype actually run? -------
def probe_sdpa_backends():
    """Try each PyTorch SDPA backend on a small causal problem, in both fp32
    (what the shared training loop uses) and fp16 (what generation uses)."""
    if device != "cuda":
        print("SDPA backend probe: running on CPU, skipping (the CUDA kernels below only exist on a GPU).")
        return
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
    except Exception as e:
        print(f"SDPA backend probe unavailable in this torch version ({type(e).__name__}).")
        return
    print(f"GPU: {torch.cuda.get_device_name(0)}, compute capability {torch.cuda.get_device_capability(0)}")
    for dtype in (torch.float32, torch.float16):
        q = torch.randn(2, 12, 256, 64, device=device, dtype=dtype)
        for name in ("FLASH_ATTENTION", "EFFICIENT_ATTENTION", "CUDNN_ATTENTION", "MATH"):
            backend = getattr(SDPBackend, name, None)
            if backend is None:
                continue
            try:
                with sdpa_kernel(backend):
                    F.scaled_dot_product_attention(q, q, q, is_causal=True)
                status = "available"
            except Exception as e:
                status = f"NOT available ({type(e).__name__})"
            print(f"  {str(dtype):15s} {name:20s} {status}")


# ---- Native GQA (enable_gqa) vs pre-expanded K/V: measure, don't guess ------
def pick_gqa_prefill_path():
    """Decides the global USE_NATIVE_GQA by timing both variants of the
    parallel attention call at the training shape, in fp32 and fp16. Native GQA
    is only chosen if it works AND is at least 10% faster in both dtypes."""
    global USE_NATIVE_GQA
    USE_NATIVE_GQA = False
    if device != "cuda" or _attn0.group_size == 1:
        print("Parallel attention path: pre-expanded K/V (repeat_interleave) -- no GPU benchmark on this device.")
        return
    seq_len, batch = CONTEXT_LEN - 1, BATCH_SIZE
    wins = []
    for dtype in (torch.float32, torch.float16):
        q = torch.randn(batch, _attn0.n_heads, seq_len, _attn0.head_dim, device=device, dtype=dtype)
        k = torch.randn(batch, _attn0.n_kv_heads, seq_len, _attn0.head_dim, device=device, dtype=dtype)
        v = torch.randn_like(k)

        def run_expanded():
            ke = k.repeat_interleave(_attn0.group_size, dim=1)
            ve = v.repeat_interleave(_attn0.group_size, dim=1)
            return F.scaled_dot_product_attention(q, ke, ve, is_causal=True)

        def run_native():
            return F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=True)

        timings = {}
        for label, fn in (("expanded", run_expanded), ("native", run_native)):
            try:
                for _ in range(3):
                    fn()
                torch.cuda.synchronize()
                start = torch.cuda.Event(enable_timing=True)
                stop = torch.cuda.Event(enable_timing=True)
                start.record()
                for _ in range(20):
                    fn()
                stop.record()
                torch.cuda.synchronize()
                timings[label] = start.elapsed_time(stop) / 20
            except Exception as e:
                timings[label] = None
                print(f"  {label} attention failed in {dtype}: {type(e).__name__}")
        if timings["expanded"] is not None and timings["native"] is not None:
            print(f"  {str(dtype):15s} pre-expanded K/V: {timings['expanded']:.2f} ms   "
                  f"enable_gqa=True: {timings['native']:.2f} ms")
            wins.append(timings["native"] < 0.9 * timings["expanded"])
        else:
            wins.append(False)
    USE_NATIVE_GQA = all(wins)
    print(f"Parallel attention path chosen: "
          f"{'enable_gqa=True (native)' if USE_NATIVE_GQA else 'pre-expanded K/V (repeat_interleave)'}")


probe_sdpa_backends()
pick_gqa_prefill_path()
log_event("model_built", params=num_params, vocab_size=vocab_size, d_model=model.token_emb.embedding_dim, n_heads=model.blocks[0].attn.n_heads,
          n_kv_heads=model.blocks[0].attn.n_kv_heads, n_layers=len(model.blocks), d_ff=model.blocks[0].ff[0].out_features, positional_encoding=POSITIONAL_ENCODING, hybrid=HYBRID_BRANCHES, checkpoint=os.path.basename(CHECKPOINT_PATH))

Number of parameters: 126,937,744 (126.9M)
Parameters per token: 126,937,744 (dense model: all of them are used)
Positional encoding: rope   context length: 2048

KV cache per sequence at 2048 tokens (fp16): GQA 12.6 MB vs MHA 50.3 MB (4x smaller)
GPU: Tesla T4, compute capability (7, 5)
  torch.float32   FLASH_ATTENTION      NOT available (RuntimeError)
  torch.float32   EFFICIENT_ATTENTION  available
  torch.float32   CUDNN_ATTENTION      NOT available (RuntimeError)


/tmp/ipykernel_23/1130455132.py:63: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:986.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/1130455132.py:63: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/1130455132.py:63: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:988.)
  F.scaled_dot_product_attention(q, q, q, is_causal=True)
/tmp/ipykernel_23/1130455132.py:63: UserWarning: Flash attention only supports gpu architectures in the range [sm80, sm121]. Attempting to run on a sm 7.5 gpu. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:342.)
  F.scaled_do

  torch.float32   MATH                 available
  torch.float16   FLASH_ATTENTION      NOT available (RuntimeError)
  torch.float16   EFFICIENT_ATTENTION  available
  torch.float16   CUDNN_ATTENTION      NOT available (RuntimeError)
  torch.float16   MATH                 available
  torch.float32   pre-expanded K/V: 7.92 ms   enable_gqa=True: 23.02 ms
  torch.float16   pre-expanded K/V: 1.43 ms   enable_gqa=True: 25.48 ms
Parallel attention path chosen: pre-expanded K/V (repeat_interleave)


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


In [22]:
def gdn_share_report():
    """After a forward pass in eval mode: how much of each layer's output comes from the GDN branch (mean softmax gate weight; 0.50 = the initial 50/50)."""
    shares = [(i, b.last_gdn_share) for i, b in enumerate(model.blocks) if getattr(b, "use_gdn", False) and b.last_gdn_share is not None]
    if not shares:
        print("GDN share report: no eval-mode forward pass has run yet (or the model has no GDN branch).")
        return
    print("Share of the GDN branch in the token-wise gate, per layer (0.50 = initial): " + "  ".join(f"L{i}:{s:.2f}" for i, s in shares))


In [23]:
if TRAINING_FINISHED:
    model.load_state_dict(_ck["model"])
    model.eval()
    final_heldout_loss, final_heldout_ppl = heldout_loss()
    print(f"Held-out Supreme Court loss of this model ({len(heldout_chunks)} chunks): {final_heldout_loss:.4f} nats/token  (perplexity {final_heldout_ppl:.2f})")
    print("For comparison: the MoE notebooks print the 160M-word MoE's loss on the same held-out chunks.")
    gdn_share_report()
    log_event("hybrid_final_heldout_loss", loss=round(final_heldout_loss, 4), ppl=round(final_heldout_ppl, 3), chunks=len(heldout_chunks))
    sync_run_log()
del _ck
import gc; gc.collect()


Held-out Supreme Court loss of this model (600 chunks): 3.3338 nats/token  (perplexity 28.04)
For comparison: the MoE notebooks print the 160M-word MoE's loss on the same held-out chunks.
Share of the GDN branch in the token-wise gate, per layer (0.50 = initial): L0:0.54  L1:0.99  L2:0.93  L3:0.90  L4:0.80  L5:0.82  L6:0.68  L7:0.41


/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

246

## 9. Generation code (KV-cached sampler, now with top-k and min-p)

In [24]:
PAD_ID = 0   # the BPE tokenizer's special_tokens list put "<pad>" first, so its id is 0
PROMPT_KEEP = CONTEXT_LEN - CACHE_HEADROOM          # max prompt-window tokens (see Section 8)
CACHE_DTYPE = torch.float16 if device == "cuda" else torch.float32   # cache stored in the dtype generation computes in
assert 0 < PROMPT_KEEP < CONTEXT_LEN, "CACHE_HEADROOM must be between 1 and CONTEXT_LEN - 1"

# ---- Decoding knobs (TEMPERATURE itself lives in the Settings cell) -----------
# Plain greedy decoding (TEMPERATURE = 0) tends to fall into loops on small models,
# so the defaults here are a starting point that fights repetition; Step 15 has a
# sweep that scores several settings on held-out Supreme Court examples so you can
# choose from numbers. For plain greedy: TEMPERATURE = 0, NO_REPEAT_NGRAM = 0.
# To mimic the official starter: TEMPERATURE = 0.1 (Settings cell), TOP_P = 0.95.
TOP_P = 0.9                # nucleus sampling, used only when TEMPERATURE > 0: sample from the
                           # smallest set of tokens whose probabilities add up to TOP_P. 1.0 = off.
REPETITION_PENALTY = 1.0   # > 1 makes tokens this continuation has ALREADY generated less likely
                           # (CTRL / Hugging Face style: positive logits divided, negative ones
                           # multiplied). Applies to generated tokens only, not the prompt, so
                           # names and terms from the prompt can still be reused. 1.0 = off.
                           # Note it hits ALL reused tokens (incl. "the", "of"), so keep it small (1.05-1.15).
NO_REPEAT_NGRAM = 4        # never let the continuation repeat any sequence of this many TOKENS
                           # (Hugging Face `no_repeat_ngram_size`, on generated tokens only). More
                           # surgical than a penalty: it only blocks exact repeats, so loops die but
                           # ordinary words can be reused. 0 = off; 4 tokens is roughly 3 words.
TOP_K = 0                  # keep only the K most likely tokens before sampling (0 = off)
MIN_P = 0.0                # drop tokens whose probability is below MIN_P x the top token's probability (0 = off)
GEN_SEED = 0               # sampling seed, set at the start of every generate_* call so a run is
                           # reproducible (only matters when TEMPERATURE > 0). None = don't seed.


def fp16_autocast():
    # T4 GPUs have dedicated fp16 tensor cores; no gradients here, so none of the
    # usual mixed-precision-training complications apply. No-op on CPU.
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda"))


class NoRepeatNgram:
    """Bans any token that would complete an n-token sequence this continuation has
    already generated (like Hugging Face's no_repeat_ngram_size, but looking only at
    the GENERATED tokens, so words and phrases can still be copied from the prompt)."""

    def __init__(self, batch_size, n):
        assert n >= 2, "NO_REPEAT_NGRAM must be 0 (off) or >= 2"
        self.n = n
        self.tokens = [[] for _ in range(batch_size)]
        self.follow = [dict() for _ in range(batch_size)]   # (n-1)-token prefix -> tokens that followed it

    def _prefix(self, toks):
        return tuple(toks[len(toks) - (self.n - 1):])

    def banned(self, vocab_size):
        """(batch, vocab_size) bool mask: True = this token would repeat an n-gram."""
        mask = torch.zeros(len(self.tokens), vocab_size, dtype=torch.bool, device=device)
        for row, toks in enumerate(self.tokens):
            if len(toks) >= self.n - 1:
                blocked = self.follow[row].get(self._prefix(toks))
                if blocked:
                    mask[row, list(blocked)] = True
        return mask

    def update(self, next_list, done):
        """Record the token each still-generating row just produced."""
        for row, token in enumerate(next_list):
            if done[row]:
                continue
            toks = self.tokens[row]
            if len(toks) >= self.n - 1:
                self.follow[row].setdefault(self._prefix(toks), set()).add(token)
            toks.append(token)


def sample_next_word_ids_batch(logits_batch, temperature, seen=None, banned=None):
    """Pick the next token for every row. logits_batch: (batch, vocab_size).
    seen: optional (batch, vocab_size) bool -- tokens each row has already
    generated (only used when REPETITION_PENALTY != 1). banned: optional
    (batch, vocab_size) bool -- tokens that may not be picked (n-gram blocking).
    Returns (batch,) ids. Order: ban n-gram repeats -> repetition penalty ->
    (temperature 0: argmax) -> temperature -> top-p filter -> sample."""
    if banned is not None:
        logits_batch = logits_batch.masked_fill(banned, float("-inf"))
    if seen is not None and REPETITION_PENALTY != 1.0:
        penalised = torch.where(logits_batch > 0, logits_batch / REPETITION_PENALTY,
                                logits_batch * REPETITION_PENALTY)
        logits_batch = torch.where(seen, penalised, logits_batch)
    if temperature == 0:
        return logits_batch.argmax(dim=-1)
    logits_batch = logits_batch / temperature
    if TOP_K > 0:
        kth = torch.topk(logits_batch, min(TOP_K, logits_batch.shape[-1]), dim=-1).values[..., -1:]
        logits_batch = logits_batch.masked_fill(logits_batch < kth, float("-inf"))
    if MIN_P > 0.0:
        _p = F.softmax(logits_batch, dim=-1)
        logits_batch = logits_batch.masked_fill(_p < MIN_P * _p.max(dim=-1, keepdim=True).values, float("-inf"))
    if TOP_P < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits_batch, descending=True, dim=-1)
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        # drop every token whose predecessors already add up to more than TOP_P
        # (so the single most likely token is always kept)
        drop = (sorted_probs.cumsum(dim=-1) - sorted_probs) > TOP_P
        sorted_logits = sorted_logits.masked_fill(drop, float("-inf"))
        logits_batch = torch.full_like(logits_batch, float("-inf")).scatter(-1, sorted_idx, sorted_logits)
    probabilities = F.softmax(logits_batch, dim=-1)
    return torch.multinomial(probabilities, num_samples=1).squeeze(-1)


def _track_seen(seen, next_ids, done):
    """Record the tokens just generated for rows that were still generating."""
    if seen is None:
        return
    rows = torch.arange(len(done), device=next_ids.device)
    still_generating = torch.tensor([not d for d in done], device=next_ids.device)
    seen[rows, next_ids] = seen[rows, next_ids] | still_generating


def _pad_windows(windows):
    """Left-pad a list of token-id lists to a common length. Returns the padded
    id tensor, the per-row pad counts (list), and the pad counts as a tensor."""
    width = max(len(w) for w in windows)
    pad_lens = [width - len(w) for w in windows]
    x = torch.tensor([[PAD_ID] * pad_lens[i] + windows[i] for i in range(len(windows))],
                     dtype=torch.long, device=device)
    return x, pad_lens, torch.tensor(pad_lens, dtype=torch.long, device=device)


def _prefill_from_history(history, cache):
    """(Re)build the cache from the last PROMPT_KEEP tokens of each row's
    history. Returns (logits for the next token, pad tensor, has_padding)."""
    x, pad_lens, pad_t = _pad_windows([h[-PROMPT_KEEP:] for h in history])
    logits = model.prefill(x, pad_lens, cache)
    return logits, pad_t, any(p > 0 for p in pad_lens)


def _finish(results, batch_indices, generated, num_words):
    for row, original_index in enumerate(batch_indices):
        words = decode(generated[row]).split()
        words = words[:num_words]
        while len(words) < num_words:   # safety net (should basically never happen)
            words.append("<unk>")
        results[original_index] = " ".join(words)


def generate_continuations_batch(prompt_texts, num_words=128, batch_size=64, verbose=True):
    """KV-cached generation. Lower batch_size if you hit a CUDA out-of-memory error."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))   # similar lengths -> little/no padding
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6   # safety cap so this can never loop forever
    attn0 = model.blocks[0].attn

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]     # prompt window + everything generated so far
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz
        cache = KVCache(len(model.blocks), bsz, attn0.n_kv_heads, CONTEXT_LEN, attn0.head_dim,
                        device, CACHE_DTYPE)

        with torch.no_grad(), fp16_autocast():
            logits, pad_t, has_padding = _prefill_from_history(history, cache)      # PREFILL
            seen = torch.zeros_like(logits, dtype=torch.bool) if REPETITION_PENALTY != 1.0 else None
            ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
            for _ in range(max_new_tokens):
                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()          # one GPU->CPU sync per step, not one per row
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    history[i].append(next_list[i])
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if cache.length >= cache.max_len:
                    # cache full: re-prefill the most recent PROMPT_KEEP tokens
                    logits, pad_t, has_padding = _prefill_from_history(history, cache)
                else:
                    logits = model.decode_step(next_ids, pad_t, has_padding, cache)   # DECODE

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  generated {min(batch_start + batch_size, len(order))} / {len(order)}")

    return results


def generate_continuations_batch_uncached(prompt_texts, num_words=128, batch_size=64, verbose=False):
    """REFERENCE: the same windowing schedule as the cached generator (same
    prompt cut, same re-window points), but recomputing the whole window with
    the ordinary parallel forward pass on every step -- no cache. Only used to
    check and time the cached version."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompt_texts)
    max_new_tokens = num_words * 6

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz

        def new_window():
            starts = [len(h) - min(len(h), PROMPT_KEEP) for h in history]
            return starts, max(len(h) - s for h, s in zip(history, starts))

        starts, width = new_window()
        seen = None
        ngram = NoRepeatNgram(bsz, NO_REPEAT_NGRAM) if NO_REPEAT_NGRAM else None
        with torch.no_grad(), fp16_autocast():
            for _ in range(max_new_tokens):
                x, pad_lens, pad_t = _pad_windows([h[s:] for h, s in zip(history, starts)])
                arange = torch.arange(x.shape[1], device=device).unsqueeze(0)
                position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)
                mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None
                logits = model(x, key_padding_mask=mask, position_ids=position_ids, last_position_only=True)
                if seen is None and REPETITION_PENALTY != 1.0:
                    seen = torch.zeros_like(logits, dtype=torch.bool)

                banned = ngram.banned(logits.shape[-1]) if ngram else None
                next_ids = sample_next_word_ids_batch(logits.float(), TEMPERATURE, seen, banned)
                _track_seen(seen, next_ids, done)
                next_list = next_ids.tolist()
                if ngram:
                    ngram.update(next_list, done)
                for i in range(bsz):
                    history[i].append(next_list[i])
                    if not done[i]:
                        generated[i].append(next_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if width >= CONTEXT_LEN:
                    starts, width = new_window()     # mirrors "cache full -> re-prefill"
                else:
                    width += 1

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  (uncached) generated {min(batch_start + batch_size, len(order))} / {len(order)}")

    return results


def generate_continuation(prompt_text, num_words=128):
    """Single-prompt convenience wrapper around the batched version above."""
    return generate_continuations_batch([prompt_text], num_words=num_words,
                                         batch_size=1, verbose=False)[0]


# quick sanity check on one example before generating all of them
example_id = list(test_prompts.keys())[0]
example_continuation = generate_continuation(test_prompts[example_id])
print("Example continuation:")
print(example_continuation)
print()
print("Word count:", len(example_continuation.split()), "(should be 128)")

Example continuation:
well settled that the delay in filing the writ petition is not fatal to the writ petition. The delay in filing writ petition is neither intentional nor deliberate or deliberate or without sufficient cause and the delay is condoned. The writ petition is liable to be dismissed on the ground of delay and laches. The writ petition fails and is dismissed. (Rajesh Bindal) vs. IN THE HIGH COURT OF HIMACHAL PRADESH, SHIMLA CWP No. 4607 of 2010 Date of decision: 18.4.2010 State of H.P. & Ors. . Petitioners Versus State of H.P. and others. . Respondents Coram The Hon'ble Mr. Justice Kurian Joseph, Chief Justice The Hon'ble Mr. JUSTICE V.K. Ahuja, J. For the petitioner: Mr. R.K. Bawa, Advocate General with Mr. J.K. Verma, Deputy Advocate General. For

Word count: 128 (should be 128)


### Beam search

Beam search keeps the most probable partial continuations (`BEAM_WIDTH` of them) for every prompt and extends them all, instead of committing to one token at a time. A continuation is finished when its 128th word ends, so all finished candidates cover the same text and their total log-probabilities are comparable. It uses the same KV cache (reordered every step so beams can pick parents), `NO_REPEAT_NGRAM` applies per beam, and temperature / top-p / top-k / min-p do not apply (beam search is deterministic). It is slower than sampling: each prompt needs `BEAM_WIDTH` rows.

In [25]:
# ---- Beam search (KV-cached, batched over prompts x beams) --------------------------------------------------------
# What it does: keeps the BEAM_WIDTH most probable partial continuations for every prompt and extends all of them one token at a time.
# A continuation is FINISHED when its 128th word ends (a token whose string ends with '</w>' ends a word), so every finished candidate
# covers exactly the same amount of text and their total log-probabilities are directly comparable (no length normalisation needed).
# The search for a prompt stops as soon as the best finished candidate is at least as probable as every unfinished beam
# (unfinished beams can only lose probability as they grow). NO_REPEAT_NGRAM is applied per beam. Temperature, top-p, top-k, min-p and the
# repetition penalty do not apply (beam search is deterministic).
BEAM_WIDTH = 1             # 1 = beam search off (the normal sampler/greedy path is used)
BEAM_PROMPT_BATCH = 16     # prompts per batch in beam mode (each prompt uses BEAM_WIDTH rows, so 16 x 4 = 64 rows)


def _beam_word_end_table():
    return torch.tensor([isinstance(t, str) and t.endswith("</w>") for t in bpe_tokenizer.itos], dtype=torch.bool, device=device)


def _ngram_banned(seq, n, vocab):
    """seq: (rows, t) generated tokens so far. Returns a (rows, vocab) bool mask of tokens that would complete an n-gram
    (n tokens) that already occurred earlier in the same row's generated tokens (same rule as NoRepeatNgram)."""
    rows, t = seq.shape
    banned = torch.zeros(rows, vocab, dtype=torch.int32, device=seq.device)
    if n < 2 or t < n:
        return banned.bool()
    prefix = seq[:, t - (n - 1):]                                  # the last n-1 tokens
    windows = seq[:, :t - 1].unfold(1, n - 1, 1)                   # every earlier window of n-1 tokens (rows, t-n+1, n-1)
    match = (windows == prefix.unsqueeze(1)).all(dim=-1)           # which earlier windows equal the current prefix
    following = seq[:, n - 1:t]                                    # the token that followed each earlier window
    banned.scatter_reduce_(1, following, match.to(torch.int32), reduce="amax", include_self=True)
    return banned.bool()


def _reorder_state(obj, index):
    """Reorder the batch dimension of every tensor inside a nested list/tuple (used for the Canon state, if the model has one)."""
    if obj is None:
        return None
    if isinstance(obj, torch.Tensor):
        return obj.index_select(0, index)
    return type(obj)(_reorder_state(o, index) for o in obj) if isinstance(obj, (list, tuple)) else obj


def _reorder_cache(cache, index):
    """Make row r of the cache a copy of row index[r] (beams choosing their parents)."""
    length = cache.length
    for layer in range(len(cache.k)):
        cache.k[layer][:, :, :length] = cache.k[layer][index, :, :length]
        cache.v[layer][:, :, :length] = cache.v[layer][index, :, :length]
    if getattr(cache, "canon", None) is not None:
        cache.canon = _reorder_state(cache.canon, index)


@torch.no_grad()
def generate_continuations_beam(prompt_texts, num_words=128, beams=None, prompt_batch=None, verbose=False):
    model.eval()
    B = beams or BEAM_WIDTH
    PB = prompt_batch or BEAM_PROMPT_BATCH
    word_end = _beam_word_end_table()
    encoded = [encode(p)[-PROMPT_KEEP:] for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompt_texts)
    attn0 = model.blocks[0].attn
    NEG = float("-inf")

    for batch_start in range(0, len(order), PB):
        idxs = order[batch_start:batch_start + PB]
        P = len(idxs)
        R = P * B
        history = [list(encoded[i]) for i in idxs for _ in range(B)]          # every prompt repeated B times
        cache = KVCache(len(model.blocks), R, attn0.n_kv_heads, CONTEXT_LEN, attn0.head_dim, device, CACHE_DTYPE)
        with fp16_autocast():
            x, pad_lens, pad_t = _pad_windows(history)
            has_padding = any(p > 0 for p in pad_lens)
            logits = model.prefill(x, pad_lens, cache)
            vocab = logits.shape[-1]
            score = torch.full((P, B), NEG, device=device)
            score[:, 0] = 0.0                                                    # only beam 0 is alive at the start (the B copies are identical)
            wc = torch.zeros(R, dtype=torch.long, device=device)                 # words finished so far, per row
            seq = torch.zeros(R, 0, dtype=torch.long, device=device)
            best_score = torch.full((P,), NEG, device=device)
            best_seq = [None] * P
            parent_base = (torch.arange(P, device=device) * B).unsqueeze(1)
            max_steps = min(num_words * 6, CONTEXT_LEN - x.shape[1] - 1)
            for step in range(max_steps):
                logp = F.log_softmax(logits.float(), dim=-1)
                if NO_REPEAT_NGRAM:
                    logp = logp.masked_fill(_ngram_banned(seq, NO_REPEAT_NGRAM, vocab), NEG)
                cand = (score.reshape(R, 1) + logp).reshape(P, B * vocab)
                top_s, top_i = cand.topk(2 * B, dim=-1)                          # best 2B (parent, token) pairs per prompt
                parent, tok = top_i // vocab, top_i % vocab
                parent_row = parent + parent_base
                new_wc = wc[parent_row] + word_end[tok].long()
                finishing = new_wc >= num_words
                # finished candidates: keep the best per prompt
                fin_s = top_s.masked_fill(~finishing, NEG)
                m, j = fin_s.max(dim=1)
                improve = m > best_score
                for p in improve.nonzero().flatten().tolist():
                    row = parent_row[p, j[p]]
                    best_seq[p] = torch.cat([seq[row], tok[p, j[p]].view(1)])
                best_score = torch.where(improve, m, best_score)
                # the next B alive beams: the best candidates that are not finished
                new_s, sel = top_s.masked_fill(finishing, NEG).topk(B, dim=-1)
                new_parent_row = parent_row.gather(1, sel).reshape(-1)
                new_tok = tok.gather(1, sel).reshape(-1)
                wc = new_wc.gather(1, sel).reshape(-1)
                score = new_s
                alive_best = score.max(dim=1).values
                finished_all = ((best_score > NEG) & (best_score >= alive_best)) | (alive_best == NEG)
                seq = torch.cat([seq[new_parent_row], new_tok.unsqueeze(1)], dim=1)
                if bool(finished_all.all()):
                    break
                _reorder_cache(cache, new_parent_row)
                logits = model.decode_step(new_tok, pad_t, has_padding, cache)
        generated = []
        for p in range(P):
            if best_seq[p] is not None:
                generated.append(best_seq[p].tolist())
            else:                                                                 # safety net: no beam reached 128 words; use the best unfinished one
                generated.append(seq[p * B + int(score[p].argmax())].tolist())
        _finish(results, idxs, generated, num_words)
        if verbose:
            print(f"  (beam {B}) generated {min(batch_start + PB, len(order))} / {len(order)}")
    return results


## 10. Local examples and the scorer

Up to `LOCAL_EVAL_EXAMPLES` windows: one random 640-word window per held-out Supreme Court document, the first 512 words as prompt and the
next 128 as the reference. Scores are reported with a **standard error for BERTScore** (spread of per-example scores / sqrt(n)): a difference
between two settings smaller than about two standard errors should be treated as noise.

In [26]:
!pip install -q bert-score rouge-score sacrebleu
import math, random, time
from bert_score import BERTScorer
from rouge_score import rouge_scorer
import sacrebleu

EVAL_SEED = 1234
LOCAL_EVAL_EXAMPLES = 1000    # windows scored in the final re-check; the first SWEEP_EXAMPLES of them are used for the wide sweep
local_prompts, local_references = [], []
sc_path = HELDOUT_FILES.get("supreme_court")
assert sc_path, "no held-out Supreme Court file -- Section 5 did not produce one"
rng = random.Random(EVAL_SEED)
with open(sc_path, "r", encoding="utf-8") as f:
    sc_docs = [line.split() for line in f]
sc_docs = [d for d in sc_docs if len(d) >= 640]
rng.shuffle(sc_docs)
# Several non-overlapping 640-word windows per document if there are fewer than LOCAL_EVAL_EXAMPLES documents. Round-robin over the
# documents, so the first windows all come from different documents.
_starts = [[] for _ in sc_docs]
_stalled = 0
while len(local_prompts) < LOCAL_EVAL_EXAMPLES and _stalled < 3:
    _added = 0
    for di, words in enumerate(sc_docs):
        if len(local_prompts) >= LOCAL_EVAL_EXAMPLES:
            break
        for _try in range(20):
            start = rng.randint(0, len(words) - 640)
            if all(abs(start - p) >= 640 for p in _starts[di]):
                _starts[di].append(start)
                window = words[start:start + 640]
                local_prompts.append(" ".join(window[:512]))
                local_references.append(" ".join(window[512:]))
                _added += 1
                break
    _stalled = _stalled + 1 if _added == 0 else 0
print(f"Built {len(local_prompts)} local evaluation windows from {len(sc_docs)} held-out Supreme Court documents "
      f"({max(len(x) for x in _starts)} windows per document at most)")

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
_bert_scorer, _rouge = None, rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def repeated_4gram_rate(texts):
    repeated = total = 0
    for text in texts:
        words, seen = text.split(), set()
        for i in range(len(words) - 3):
            gram = tuple(words[i:i + 4])
            total += 1
            repeated += gram in seen
            seen.add(gram)
    return repeated / max(1, total)


def score_local(predictions, references):
    global _bert_scorer
    if _bert_scorer is None:
        _bert_scorer = BERTScorer(lang="en", batch_size=16)
    _, _, f1 = _bert_scorer.score(predictions, references)
    f1 = f1.float()
    rouge_l = [_rouge.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]
    return {"BERTScore F1": f1.mean().item(),
            "BERT SE": (f1.std().item() / math.sqrt(len(f1))) if len(f1) > 1 else 0.0,
            "ROUGE-L": sum(rouge_l) / len(rouge_l),
            "BLEU-4": sacrebleu.corpus_bleu(predictions, [references]).score,
            "repeated 4-grams": repeated_4gram_rate(predictions),
            "_f1": f1.tolist()}


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 22.3 MB/s eta 0:00:00
Built 1000 local evaluation windows from 609 held-out Supreme Court documents (2 windows per document at most)


## 13. Decoding settings and helpers

In [27]:
GEN_BATCH_SIZE = 64      # used by greedy/sampling/MBR; lower if you hit a CUDA out-of-memory error
MBR_SAMPLES = 6
_BASE = dict(TEMPERATURE=0.0, TOP_P=1.0, TOP_K=0, MIN_P=0.0, REPETITION_PENALTY=1.0, NO_REPEAT_NGRAM=0, BEAM_WIDTH=1)
DECODINGS = {
    # beam search, width 4, never repeat a 4-token sequence (the best row of the earlier decoding sweep on 300 windows)
    "beam": dict(name="beam 4 + ng4", slug="beam4_ng4", mbr=False, knobs=dict(_BASE, NO_REPEAT_NGRAM=4, BEAM_WIDTH=4)),
    # MBR x6: 6 samples (T 0.7, top-p 0.9, no-repeat 4), keep the one most similar (ROUGE-L) to the others
    "mbr":  dict(name="MBR x6 (T0.7 p0.9 + ng4)", slug="MBR_x6_T0_7_p0_9_ng4", mbr=True,
                 knobs=dict(_BASE, TEMPERATURE=0.7, TOP_P=0.9, NO_REPEAT_NGRAM=4)),
}

KNOB_NAMES = list(_BASE)


def apply_knobs(cfg):
    global TEMPERATURE, TOP_P, TOP_K, MIN_P, REPETITION_PENALTY, NO_REPEAT_NGRAM, BEAM_WIDTH
    TEMPERATURE, TOP_P, TOP_K, MIN_P = cfg["TEMPERATURE"], cfg["TOP_P"], cfg["TOP_K"], cfg["MIN_P"]
    REPETITION_PENALTY, NO_REPEAT_NGRAM = cfg["REPETITION_PENALTY"], cfg["NO_REPEAT_NGRAM"]
    BEAM_WIDTH = cfg["BEAM_WIDTH"]


def generate_mbr(prompts, k, num_words=128):
    """k sampled continuations per prompt (seeds GEN_SEED, GEN_SEED+1, ...); keep the one whose average ROUGE-L
    against the other k-1 is highest. Uses whatever decoding knobs are currently set."""
    global GEN_SEED
    base_seed = GEN_SEED if GEN_SEED is not None else 0
    pools = []
    try:
        for j in range(k):
            GEN_SEED = base_seed + j
            pools.append(generate_continuations_batch(prompts, num_words=num_words, batch_size=GEN_BATCH_SIZE, verbose=False))
    finally:
        GEN_SEED = base_seed
    chosen = []
    for i in range(len(prompts)):
        cands = [pools[j][i] for j in range(k)]
        agree = [sum(_rouge.score(cands[b], cands[a])["rougeL"].fmeasure for b in range(k) if b != a) / (k - 1) for a in range(k)]
        chosen.append(cands[max(range(k), key=agree.__getitem__)])
    return chosen


def generate_current(prompts, mbr=False, verbose=False):
    """Generate with whatever decoding knobs are currently set: MBR, beam search (BEAM_WIDTH > 1) or the normal sampler/greedy."""
    if mbr:
        return generate_mbr(prompts, MBR_SAMPLES)
    if BEAM_WIDTH > 1:
        return generate_continuations_beam(prompts, num_words=128, verbose=verbose)
    return generate_continuations_batch(prompts, num_words=128, batch_size=GEN_BATCH_SIZE, verbose=verbose)




## 14. Local score of this model with beam search (held-out Supreme Court windows)

Same windows and decoding as the earlier sweep's 1000-window re-check of the base MoE checkpoint (BERTScore F1 0.8313 for beam 4 + ng4).

In [28]:
if TRAINING_FINISHED:
    _cfg = DECODINGS["beam"]
    N_LOCAL = 30 if QUICK_TEST else 1000          # the held-out windows scored (the earlier sweep used the same 1000)
    apply_knobs(_cfg["knobs"])
    t0 = time.time()
    _preds = generate_current(local_prompts[:N_LOCAL], mbr=False)
    _res = score_local(_preds, local_references[:N_LOCAL])
    print(f"hybrid (attention + GDN) model, {_cfg['name']} on {N_LOCAL} held-out Supreme Court windows ({(time.time() - t0) / 60:.1f} min):")
    print(f"  BERTScore F1 {_res['BERTScore F1']:.4f} (+-{_res['BERT SE']:.4f})   ROUGE-L {_res['ROUGE-L']:.4f}   BLEU-4 {_res['BLEU-4']:.2f}   repeated 4-grams {_res['repeated 4-grams']:.4f}")
    print("  BASE MoE checkpoint, same beam 4 + ng4, 1000 windows (decoding sweep): BERTScore F1 0.8313 (+-0.0006)   ROUGE-L 0.2266   BLEU-4 7.73")
    if QUICK_TEST:
        print("  (quick test: only 30 windows, so this is not comparable)")
    log_event("hybrid_local_score", config=_cfg["name"], windows=N_LOCAL, **{k: v for k, v in _res.items() if k != "_f1"})
    sync_run_log()
else:
    print('Training is not finished yet -- run the notebook again to continue.')


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (4 outputs left out)

/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 15. Generate the real test continuations and write the submission (beam search)

In [29]:
if TRAINING_FINISHED:
    FINAL_DECODINGS = ["beam"]       # ["beam"] = beam 4 + no-repeat-4 (default). Add "mbr" to ALSO write an MBR x6 submission, e.g. ["beam", "mbr"].

    test_ids = list(test_prompts.keys())
    if QUICK_TEST:
        test_ids = test_ids[:16]
    test_texts = [test_prompts[i] for i in test_ids]
    import re as _re
    for key in FINAL_DECODINGS:
        cfg = DECODINGS[key]
        apply_knobs(cfg["knobs"])
        print(f"Generating the test continuations with: {cfg['name']}  {cfg['knobs']}  MBR={cfg['mbr']}")
        t0 = time.time()
        final_texts = generate_current(test_texts, mbr=cfg["mbr"], verbose=True)
        print(f"generated {len(final_texts)} continuations in {(time.time() - t0) / 60:.1f} min")

        predictions = {doc_id: {"text": text} for doc_id, text in zip(test_ids, final_texts)}
        prediction_path = "/kaggle/working/prediction.json"
        with open(prediction_path, "w") as f:
            json.dump(predictions, f, indent=2)
        bad = [d for d, p in predictions.items() if len(p["text"].split()) != 128]
        assert not bad, f"{len(bad)} predictions do not have exactly 128 words: {bad[:5]}"
        if not QUICK_TEST:
            assert set(predictions) == set(test_prompts), "prediction ids differ from the test ids"
        submission_path = f"/kaggle/working/submission_hybrid_gdn_{cfg['slug']}{'_QUICKTEST' if QUICK_TEST else ''}.zip"
        with zipfile.ZipFile(submission_path, "w") as zf:
            zf.write(prediction_path, arcname="prediction.json")     # prediction.json must be at the ROOT of the zip
        print(f"All checks passed. Saved {submission_path}  (download it from the Output panel and upload it to Codabench)")
        log_event("hybrid_submission", config=cfg["name"], knobs=cfg["knobs"], mbr=cfg["mbr"], zip=os.path.basename(submission_path), quick_test=QUICK_TEST)
        sync_run_log()
else:
    print('Training is not finished yet -- run the notebook again to continue; no submission was written.')


Generating the test continuations with: beam 4 + ng4  {'TEMPERATURE': 0.0, 'TOP_P': 1.0, 'TOP_K': 0, 'MIN_P': 0.0, 'REPETITION_PENALTY': 1.0, 'NO_REPEAT_NGRAM': 4, 'BEAM_WIDTH': 4}  MBR=False
  (beam 4) generated 16 / 1000
  (beam 4) generated 32 / 1000
  (beam 4) generated 48 / 1000
  (beam 4) generated 64 / 1000
  (beam 4) generated 80 / 1000
  (beam 4) generated 96 / 1000
  (beam 4) generated 112 / 1000
  (beam 4) generated 128 / 1000
  (beam 4) generated 144 / 1000
  (beam 4) generated 160 / 1000
  (beam 4) generated 176 / 1000
  (beam 4) generated 192 / 1000
  (beam 4) generated 208 / 1000
  (beam 4) generated 224 / 1000
  (beam 4) generated 240 / 1000
  (beam 4) generated 256 / 1000
  (beam 4) generated 272 / 1000
  (beam 4) generated 288 / 1000
  (beam 4) generated 304 / 1000
  (beam 4) generated 320 / 1000
  (beam 4) generated 336 / 1000
  (beam 4) generated 352 / 1000
  (beam 4) generated 368 / 1000
  (beam 4) generated 384 / 1000
  (beam 4) generated 400 / 1000
  (beam 4) gen

/tmp/ipykernel_23/4292410765.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            